<a href="https://colab.research.google.com/github/gano0802/BirdCLEF2026/blob/main/BC2026_SelfKD_SED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !unzip -q /content/drive/MyDrive/kaggle/Birdclef2026/input/birdclef-2026-waveform-cache.zip -d /content

In [ ]:
# !rsync -ah --progress /content/drive/MyDrive/kaggle/Birdclef2026/input/birdclef-2026-waveform-cache.zip /content

In [ ]:
# import zipfile
# import torch
# import os
# import io
# from pathlib import Path
# from tqdm import tqdm

# # ---- 設定 ------------------------------------------------------------------
# # KaggleからダウンロードしたZipファイルのパス
# ZIP_PATH = '/content/birdclef-2026-waveform-cache.zip'

# # チャンク化したファイルの保存先（Google Driveのフォルダを指定）
# OUTPUT_DIR = '/content/drive/MyDrive/kaggle/Birdclef2026/input'
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # 1つのチャンク（塊）にまとめるファイル数
# BATCH_SIZE = 5000
# # ------------------------------------------------------------------------------

# def main():
#     with zipfile.ZipFile(ZIP_PATH, 'r') as z:
#         # Zip内の .pt ファイルのリストを取得
#         pt_files = [f for f in z.namelist() if f.endswith('.pt')]
#         print(f"Zip内の対象ファイル数: {len(pt_files):,}件")

#         # バッチ処理
#         for i in range(0, len(pt_files), BATCH_SIZE):
#             batch_list = pt_files[i : i + BATCH_SIZE]
#             chunk_data = {}
#             chunk_idx = i // BATCH_SIZE + 1

#             print(f"チャンク {chunk_idx} を処理中...")
#             for file_path in tqdm(batch_list, leave=False):
#                 # 1. Zipから対象ファイルをバイナリとしてメモリに読み込む（解凍しない！）
#                 with z.open(file_path) as f:
#                     file_bytes = f.read()

#                 # 2. BytesIOを使ってメモリ上のバイナリから直接PyTorchテンソルをロード
#                 tensor = torch.load(io.BytesIO(file_bytes), map_location='cpu', weights_only=False)

#                 # 3. ファイル名（例: "000001.pt"）をキーにして辞書に格納
#                 filename = Path(file_path).name
#                 chunk_data[filename] = tensor

#             # 4. まとまった辞書を1つの大きな .pt ファイルとしてDriveに直接保存
#             chunk_filename = os.path.join(OUTPUT_DIR, f"chunk_{chunk_idx:03d}.pt")
#             torch.save(chunk_data, chunk_filename)
#             print(f"保存完了: {chunk_filename} (格納数: {len(chunk_data)})\n")

# if __name__ == "__main__":
#     main()

In [ ]:
# import pandas as pd
# import zipfile
# import os
# from pathlib import Path

# # ---- 設定 ------------------------------------------------------------------
# ZIP_PATH = '/content/birdclef-2026-waveform-cache.zip'
# OUTPUT_DIR = '/content/drive/MyDrive/kaggle/Birdclef2026/input/'
# BATCH_SIZE = 5000

# CSV_FILES = [
#     '/content/drive/MyDrive/kaggle/Birdclef2026/input/birdclef-2026-waveform-cache/waveform_cache/audio_cache_meta.csv',
#     '/content/drive/MyDrive/kaggle/Birdclef2026/input/birdclef-2026-waveform-cache/waveform_cache/pseudo_window_meta.csv',
#     '/content/drive/MyDrive/kaggle/Birdclef2026/input/birdclef-2026-waveform-cache/waveform_cache/soundscape_cache_meta.csv'
# ]
# # ------------------------------------------------------------------------------

# # 1. Zipから「ファイル名 -> チャンク名」のマップを作る
# chunk_map = {}
# with zipfile.ZipFile(ZIP_PATH, 'r') as z:
#     pt_files = [f for f in z.namelist() if f.endswith('.pt')]

#     for i in range(0, len(pt_files), BATCH_SIZE):
#         batch_list = pt_files[i : i + BATCH_SIZE]
#         chunk_idx = i // BATCH_SIZE + 1
#         chunk_name = f"chunk_{chunk_idx:03d}.pt"

#         for file_path in batch_list:
#             # ★修正ポイント: パス全体ではなく、ファイル名(例: audio_000000.pt)だけを抽出してキーにする
#             base_name = Path(file_path).name
#             chunk_map[base_name] = chunk_name

# print("マップ作成完了。CSVを更新します...\n")
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # 2. CSVを書き換える
# for csv_file in CSV_FILES:
#     try:
#         df = pd.read_csv(csv_file)

#         # ★修正ポイント: CSVのパスからもファイル名だけを抽出する
#         # 例: "audio/audio_000000.pt" -> "audio_000000.pt"
#         df['dict_key'] = df['cache_file'].apply(lambda x: Path(x).name)

#         # ファイル名同士でチャンク名を割り当てる
#         df['chunk_file'] = df['dict_key'].map(chunk_map)

#         # エラーチェック
#         missing_count = df['chunk_file'].isna().sum()
#         if missing_count > 0:
#             print(f"⚠️ {csv_file}: {missing_count} 件見つかりません。")
#         else:
#             print(f"✅ {csv_file}: 全件正常にマッチしました！")

#         # 古い列を削除して保存
#         df.drop(columns=['cache_file'], inplace=True)
#         out_path = os.path.join(OUTPUT_DIR, f"updated_{csv_file.split('/')[-1]}")
#         df.to_csv(out_path, index=False)

#     except FileNotFoundError:
#         print(f"❌ {csv_file} が見つかりません。")

# print("\n処理が完了しました！")

In [ ]:
import os
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

SOURCE_DIR = Path('/content/drive/MyDrive/kaggle/Birdclef2026/input/')
DEST_DIR = Path('/content/local_chunk_cache/')
DEST_DIR.mkdir(parents=True, exist_ok=True)

# コピー対象のファイルリストを取得
files_to_copy = list(SOURCE_DIR.glob('chunk_*.pt')) + list(SOURCE_DIR.glob('updated_*.csv'))

def copy_file(file_path):
    dest_path = DEST_DIR / file_path.name
    # 既に存在してサイズが同じならスキップ（再実行時の高速化）
    if dest_path.exists() and dest_path.stat().st_size == file_path.stat().st_size:
        return
    shutil.copy2(file_path, dest_path)

# 8スレッドで一気に並列コピー
with ThreadPoolExecutor(max_workers=8) as executor:
    list(tqdm(executor.map(copy_file, files_to_copy), total=len(files_to_copy), desc="Copying to local disk"))

print("コピー完了！")

# S1 -- IMPORTS + CONFIG

In [ ]:
EXP = "exp30" # "exp18"

In [ ]:
# =================================================================
# S1 -- IMPORTS + CONFIG
# =================================================================
import os, sys, time, json, pickle, gc, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import torchaudio
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.special import expit as sigmoid_np
from tqdm.notebook import tqdm, trange
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True # False
torch.backends.cudnn.benchmark = False # True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name()}")

# ------------------------------------------------------------------
# Notebook Mode
# -----------------------------------------------------------------
MODE = "train"  # "train" or "infer"

if MODE == 'train':    #Kaggle struggles with full training pipeline
    DEBUG = False
else:
    DEBUG = False

# ------------------------------------------------------------------
# Paths -- Kaggle dataset layout
# -----------------------------------------------------------------
COMP_DIR = Path("/content/drive/MyDrive/kaggle/Birdclef2026")
WAVEFORM_CACHE_DIR = Path("/content/waveform_cache")
PERCH_ONNX_PATH = Path("/content/drive/MyDrive/kaggle/Birdclef2026/input/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx")

LABELS_PATH     = COMP_DIR / "input" / "train_soundscapes_labels.csv"
TAXONOMY_PATH   = COMP_DIR / "input" / "taxonomy.csv"
SAMPLE_SUB_PATH = COMP_DIR / "input" / "sample_submission.csv"
TEST_DIR        = COMP_DIR / "input" / "test_soundscapes"

OUT_DIR = Path("/content/drive/MyDrive/kaggle/Birdclef2026/outputs")

NUM_CLASSES = 234
SR = 32000

# --- Duration ---
TRAIN_DURATION = 5    # seconds
VAL_DURATION   = 5    # always 5s for competition eval
TRAIN_SAMPLES  = SR * TRAIN_DURATION
VAL_SAMPLES    = SR * VAL_DURATION

N_FOLDS = 5

# --- Mel spectrogram ---
N_FFT      = 2048
HOP_LENGTH = 512
N_MELS     = 256
FMIN       = 20
FMAX       = 16000

# --- Model ---
BACKBONE_NAME = "regnety_008.pycls_in1k"
# 'tf_efficientnetv2_s.in21k_ft_in1k' stage1: exp19
# "tf_efficientnet_b0.ns_jft_in1k" stage0: exp03
# "seresnext26t_32x4d", stage1: exp17
# "regnety_008.pycls_in1k" stage1: exp18
#  "tf_efficientnetv2_b3.in21k_ft_in1k"

# --- Perch distillation ---
USE_PERCH_DISTILL = False # True
PERCH_EMBED_DIM   = 1536
ALPHA_DISTILL     = 1.0   # MSE loss weight

# --- Training ---
FOLDS  = [1, 2, 3, 4] # [0, 1, 2, 3, 4]
EPOCHS = 25
BATCH  = 16 if DEBUG else 64        # used 64 locally, 16 for Kaggle
LR     = 5e-4
MIN_LR = 1e-5
WD     = 1e-4
WARMUP_EPOCHS = 5 # 2

# --- Upsampling ---
MIN_SAMPLE = 20

# --- Augmentation ---
AUG_PROB = 0.5
AUG_GAIN_DB_RANGE      = (-6.0, 6.0)
AUG_NOISE_SNR_DB_RANGE = (10.0, 30.0)
AUG_SHIFT_SAMPLES_MAX  = int(SR * 0.5)

# --- MixUp ---
USE_FOCAL_MIXUP    = True
MIXUP_PROB         = 0.5
MIXUP_ALPHA        = 0.4
MIXUP_HARD         = True    # union labels (hard) vs weighted blend (soft)

USE_FOCAL_SC_MIXUP     = False # True
FOCAL_SC_MIXUP_PROB    = 0.5
FOCAL_SC_MIXUP_ALPHA   = 0.4

# --- FreqMixStyle (disabled by default) ---
FREQ_MIXSTYLE_PROB  = 0.0
FREQ_MIXSTYLE_ALPHA = 0.1

# --- SpecAugment ---
FREQ_MASK_PARAM = 10
TIME_MASK_PARAM = 10
NUM_FREQ_MASKS  = 1
NUM_TIME_MASKS  = 2

# --- Source weights ---
USE_FOCAL           = True
USE_FOCAL_SECONDARY = True
USE_LABELED_SC      = True

ACTIVE_SOURCES = ["focal", "sc", "ulsc"]
SHARES = SHARES = {"focal": 0.80, "sc": 0.10, "ulsc": 0.10} # {"focal": 0.8, "sc": 0.05, "ulsc": 0.15} # {"focal": 0.8, "sc": 0.1, "ulsc": 0.01}
SOURCE_WEIGHTS = {
    "focal":         1.0,
    "focal_missing": 0.0,
    "sc":            1.0,
    "ulsc":          1.0,  # 新規追加
}

# --- Pseudo label ---
USE_PSEUDO_LABEL = True
USE_UNLABELED_SC    = True  # 新規追加

print(f"Backbone: {BACKBONE_NAME}")
print(f"Train duration: {TRAIN_DURATION}s | Mel: {N_MELS} mels, n_fft={N_FFT}, hop={HOP_LENGTH}")
print(f"Distillation: {'ON' if USE_PERCH_DISTILL else 'OFF'} (alpha={ALPHA_DISTILL})")
print(f"Self Distillation: {'ON' if USE_PSEUDO_LABEL else 'OFF'}")
print(f"Batch: {BATCH} | Epochs: {EPOCHS} | Folds: {FOLDS}")

In [ ]:
os.makedirs(OUT_DIR / f"{EXP}", exist_ok=True)

In [ ]:
if MODE == "train":
    !uv pip install -q timm torchaudio onnxscript onnx onnxruntime-gpu

# S2 -- LOAD DATA

In [ ]:
# =================================================================
# S2 -- LOAD DATA
# =================================================================

WAVEFORM_CACHE_DIR = Path("/content/drive/MyDrive/kaggle/Birdclef2026/input/birdclef-2026-waveform-cache/waveform_cache")

# --- Label ordering from sample_submission (defines column order) ---
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
taxonomy = pd.read_csv(TAXONOMY_PATH)
label_to_taxon = dict(zip(taxonomy["primary_label"].astype(str),
                          taxonomy["class_name"].astype(str)))
TAXON_MASKS = {t: np.array([i for i, l in enumerate(PRIMARY_LABELS)
                            if label_to_taxon.get(l, "") == t])
               for t in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]}

# --- Focal recording metadata ---
# audio_cache_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "audio_cache_meta.csv")
audio_cache_meta = pd.read_csv("/content/local_chunk_cache/updated_audio_cache_meta.csv")
train_df = pd.read_csv(COMP_DIR / "input" / "train.csv")
audio_cache_meta = audio_cache_meta.merge(
    train_df[["filename", "secondary_labels"]], on="filename", how="left"
)
audio_cache_meta = audio_cache_meta[
    audio_cache_meta["primary_label"].isin(LABEL2IDX)
].reset_index(drop=True)
print(f"Focal audio cache: {len(audio_cache_meta)} entries")

# --- Soundscape window metadata ---
# sc_cache_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_cache_meta.csv")
sc_cache_meta = pd.read_csv("/content/local_chunk_cache/updated_soundscape_cache_meta.csv")
sc_cache_meta["label_list"] = sc_cache_meta["label_list"].apply(
    lambda x: x.split(";") if isinstance(x, str) else []
)
print(f"Soundscape cache: {len(sc_cache_meta)} windows")

# --- Build soundscape label matrix from ground truth ---
sc_labels_raw = pd.read_csv(LABELS_PATH).drop_duplicates()
sc_labels_raw["start_sec"] = pd.to_timedelta(sc_labels_raw["start"]).dt.total_seconds().astype(int)

Y_SC = np.zeros((len(sc_cache_meta), NUM_CLASSES), dtype=np.float32)
for i, row in sc_cache_meta.iterrows():
    matches = sc_labels_raw[
        (sc_labels_raw["filename"] == row["filename"]) &
        (sc_labels_raw["start_sec"] == row["start_sec"])
    ]
    for _, m in matches.iterrows():
        for lbl in str(m["primary_label"]).split(";"):
            lbl = lbl.strip()
            if lbl in LABEL2IDX:
                Y_SC[i, LABEL2IDX[lbl]] = 1.0

labeled_sc_mask = Y_SC.sum(axis=1) > 0
print(f"Soundscape labels: {labeled_sc_mask.sum()}/{len(Y_SC)} windows labeled, "
      f"{int(Y_SC.sum())} positives, {int((Y_SC.sum(axis=0) > 0).sum())} species")

# =================================================================
# FOLD ASSIGNMENT
# =================================================================

# --- Focal: StratifiedKFold by species ---
audio_for_split = audio_cache_meta.drop_duplicates("original_idx").reset_index(drop=True)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
audio_for_split["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(audio_for_split, audio_for_split["primary_label"])):
    audio_for_split.loc[val_idx, "fold"] = fold
audio_cache_meta = audio_cache_meta.merge(
    audio_for_split[["original_idx", "fold"]], on="original_idx", how="left"
)
print(f"\nFocal fold distribution:\n{audio_cache_meta['fold'].value_counts().sort_index()}")

# --- Soundscape: file-level folds, all 66 files distributed ---
# All files including S22 participate in CV for maximum species coverage
# (46 multi-fold species vs 32 with S22 holdout). The non_s22_mask_sc
# filter in evaluation still excludes S22 windows from the primary metric.
from sklearn.model_selection import GroupKFold

sc_files = sc_cache_meta[["filename", "site"]].drop_duplicates().reset_index(drop=True)
gkf = GroupKFold(n_splits=N_FOLDS)
sc_files["fold"] = -1
for fold, (_, val_idx) in enumerate(gkf.split(sc_files, groups=sc_files["filename"])):
    sc_files.loc[sc_files.index[val_idx], "fold"] = fold

file_to_fold = dict(zip(sc_files["filename"], sc_files["fold"]))
sc_cache_meta["fold"] = sc_cache_meta["filename"].map(file_to_fold).fillna(-1).astype(int)
print(f"\nSoundscape fold distribution:")
print(sc_cache_meta["fold"].value_counts().sort_index())
# =================================================================
# UPSAMPLE RARE SPECIES
# =================================================================
counts = audio_cache_meta["primary_label"].value_counts()
rare_species = counts[counts < MIN_SAMPLE].index
extra_rows = []
for sp in rare_species:
    sp_rows = audio_cache_meta[audio_cache_meta["primary_label"] == sp]
    n_copies = int(np.ceil(MIN_SAMPLE / len(sp_rows))) - 1
    for _ in range(n_copies):
        extra_rows.append(sp_rows)

n_before = len(audio_cache_meta)
if extra_rows:
    audio_cache_meta = pd.concat([audio_cache_meta] + extra_rows, ignore_index=True)
print(f"\nUpsampled {len(rare_species)} rare species (min={MIN_SAMPLE}): "
      f"{n_before} -> {len(audio_cache_meta)} samples")

# Non-S22 mask for evaluation (S22 is a site with known label noise)
sc_sites = sc_cache_meta["site"].values
non_s22_mask_sc = sc_sites != "S22"
print(f"S22: {(~non_s22_mask_sc).sum()}, non-S22: {non_s22_mask_sc.sum()}")
print("OK Data loaded")

In [ ]:
# メタデータの読み込み
# meta_df = pd.read_csv(WAVEFORM_CACHE_DIR / "pseudo_window_meta.csv")
meta_df = pd.read_csv("/content/local_chunk_cache/updated_pseudo_window_meta.csv")

# True / False で分割してカウント
num_true = meta_df[meta_df["has_expert_label"] == True].shape[0]
num_false = meta_df[meta_df["has_expert_label"] == False].shape[0]

print("=== 全体データ数 ===")
print(f"ScDS候補 (Expert Labelあり): {num_true} 件")
print(f"ULScDS候補 (Expert Labelなし): {num_false} 件")
print(f"合計: {len(meta_df)} 件")

In [ ]:
if DEBUG:
    EPOCHS = 7
    FOLDS = [0]
    audio_cache_meta = audio_cache_meta.groupby("primary_label").head(3).reset_index(drop=True)
    sc_cache_meta = sc_cache_meta.head(50)
    Y_SC = Y_SC[:50]
    non_s22_mask_sc = non_s22_mask_sc[:50]
    print(f"DEBUG MODE: {len(audio_cache_meta)} focal, {len(sc_cache_meta)} sc, "
          f"{EPOCHS} epoch, folds={FOLDS}")

# S3 — Model Architecture

* SED (Sound Event Detection) head design  
Instead of Global Average Pooling (GAP), which dilutes a brief vocalization across the full 5-second window, the SED head makes per-frame predictions and aggregates them via learned attention weights. A species that calls for 0.3 seconds gets a sharp attention spike at those frames, rather than being averaged with 4.7 seconds of background.

* Distillation head  
A separate branch applies GAP to the raw backbone features and projects them to Perch’s 1536-d embedding space. This branch receives all backbone gradients via MSE loss against frozen Perch embeddings. The SED classification head operates on h.detach() — it only shapes the attention and frame-level classifier, never pulling the backbone away from Perch’s representation.

* Why stop-gradient?  
Without the stop-gradient, the classification loss and distillation loss would fight over the backbone’s feature representation. With it, the backbone is purely a “Perch student” — learning to reproduce Perch’s rich 14,795-species embedding geometry. The SED head then learns to classify using those frozen-quality features.

* Two model variants are implemented:  

    * V1: Simple frequency-mean pooling + attention block
    * V2 (default): Generalized Mean (GeM) frequency pooling + 512-d bottleneck + attention block — inspired by the BirdCLEF 2021 first-place solution

In [ ]:
# =================================================================
# S3 -- EVAL UTILITIES + MEL TRANSFORM + SED MODELS
# =================================================================

def compute_macro_auc(y_true, y_pred, mask=None, class_mask=None):
    """Macro-averaged AUC across evaluable species."""
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    if class_mask is not None:
        y_true, y_pred = y_true[:, class_mask], y_pred[:, class_mask]
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        try:
            aucs.append(roc_auc_score(col, y_pred[:, c]))
        except ValueError:
            continue
    return (np.mean(aucs) if aucs else float("nan")), len(aucs)

def full_eval(y_true, y_pred, ns22, tm):
    r = {}
    a, n = compute_macro_auc(y_true, y_pred)
    r["macro_auc_all"], r["n_all"] = round(a, 4), n
    a, n = compute_macro_auc(y_true, y_pred, mask=ns22)
    r["non_s22_macro"], r["n_ns22"] = round(a, 4), n
    for t, cm in tm.items():
        a, n = compute_macro_auc(y_true, y_pred, mask=ns22, class_mask=cm)
        r[f"non_s22_{t}"] = round(a, 4)
    return r

# ------------------------------------------------------------------
# GPU Mel Spectrogram
# ------------------------------------------------------------------
class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0,
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)

    def forward(self, waveform):
        return self.db_transform(self.mel_spec(waveform))

# ------------------------------------------------------------------
# GPU SpecAugment
# # ------------------------------------------------------------------
class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=TIME_MASK_PARAM)

    def forward(self, mel):
        for _ in range(NUM_FREQ_MASKS):
            mel = self.freq_mask(mel)
        for _ in range(NUM_TIME_MASKS):
            mel = self.time_mask(mel)
        return mel

# ------------------------------------------------------------------
# Frozen Perch teacher -- ONNX inference, no gradients
# ------------------------------------------------------------------
import onnxruntime as ort

class PerchTeacher:
    """Frozen Perch v2 via ONNX. Takes 5s waveforms, returns 1536-d embeddings.
    The teacher is never updated -- it provides a stable distillation target."""

    def __init__(self, onnx_path, device_str="cuda"):
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] \
            if device_str == "cuda" else ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self._out_names = [o.name for o in self.session.get_outputs()]
        self._embed_idx = None
        for i, o in enumerate(self.session.get_outputs()):
            if o.shape and o.shape[-1] == PERCH_EMBED_DIM:
                self._embed_idx = i
                break
        if self._embed_idx is None:
            self._embed_idx = 1
        print(f"Perch ONNX loaded: embed_idx={self._embed_idx}")

    @torch.no_grad()
    def embed(self, waveforms_5s):
        """waveforms_5s: (B, 160000) float32, returns (B, 1536) embeddings."""
        wav_np = waveforms_5s.cpu().numpy()
        results = self.session.run(None, {self.input_name: wav_np})
        return torch.from_numpy(results[self._embed_idx]).float()

# ------------------------------------------------------------------
# Distillation head: GAP + Linear to Perch embedding space
# ------------------------------------------------------------------
class DistillHead(nn.Module):
    """Projects backbone features to Perch's 1536-d space via GAP + Linear."""
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)

    def forward(self, feature_map):
        gap = feature_map.mean(dim=[2, 3])   # (B, C, F, T) -> (B, C)
        return self.proj(gap)                 # (B, embed_dim)

# ------------------------------------------------------------------
# SED Model V2: GeMFreq + bottleneck + AttBlock (recommended)
# ------------------------------------------------------------------
class GeMFreqPool(nn.Module):
    """Generalized Mean pooling over frequency. Learnable p starts at 3.0
    (sharper than mean, softer than max). Lets the model emphasize
    frequency bands where species vocalize."""
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)

class BirdSEDModel(nn.Module):
    """SED model with 1st-place-inspired design: https://www.kaggle.com/code/nikitababich/birdclef2025-1st-place-inference
    - GeMFreq pooling (learnable, sharper than mean)
    - 512-dim bottleneck with dropout
    - Attention-weighted clip logits from frame logits
    - Distillation: GAP+Linear branch for MSE to Perch
    - Stop gradient: backbone trains from distillation only
    """
    def __init__(self, backbone_name=BACKBONE_NAME, num_classes=NUM_CLASSES,
                 drop_path_rate=0.15, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, in_chans=1,
            num_classes=0, global_pool="", drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            n_tf = TRAIN_SAMPLES // HOP_LENGTH + 1
            dummy = torch.randn(1, 1, N_MELS, n_tf)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]
            print(f"V2 backbone: {tuple(feat.shape)}  (C={self.backbone_dim})")

        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        self.att.bias.data.fill_(0.)
        self.cla.bias.data.fill_(0.)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        distill_emb = None
        if return_distill and hasattr(self, 'distill_head'):
            # h_for_distill = F.dropout(h, p=0.3, training=self.training) # add
            distill_emb = self.distill_head(h)
            # distill_emb = self.distill_head(h_for_distill)

        # Stop gradient: SED head doesn't update the backbone
        h_cls = h.detach() if USE_PERCH_DISTILL else h

        h_cls = self.gem_freq(h_cls)            # (B, C, T)
        h_cls = h_cls.permute(0, 2, 1)          # (B, T, C)
        h_cls = self.dense(h_cls)               # (B, T, 512)
        h_cls = h_cls.permute(0, 2, 1)          # (B, 512, T)

        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)

        fw = framewise_logits.permute(0, 2, 1) if return_framewise else None
        if return_framewise and return_distill: return clip_logits, fw, distill_emb
        elif return_framewise: return clip_logits, fw
        elif return_distill: return clip_logits, distill_emb
        return clip_logits

def make_model(MODEL_NAME=BACKBONE_NAME):
        return BirdSEDModel(MODEL_NAME).to(device)

print("OK Model definitions ready")

# S4 — Data Pipeline

* Waveform loading¶  
The waveform cache stores audio as int16 PyTorch tensors (half the size of float32). Each focal recording and soundscape window is pre-sliced and saved as a separate .pt file, indexed by the metadata CSVs.

* Augmentation pipeline  
Three simple waveform augmentations applied with probability 0.5 each:

    1. Gain jitter (±6 dB) — simulates varying recording distances
    2. Additive noise (10–30 dB SNR) — simulates environmental noise
    3. Time shift (±0.5s) — simulates temporal misalignment
    4. MixUp strategy
* Two MixUp variants run independently:  

    * Focal-Focal MixUp: Blends two focal recordings with Beta(0.4, 0.4) weighting. Labels are unioned (hard MixUp).
    * Focal-Soundscape MixUp: Blends a focal recording with a labeled soundscape window, bridging the domain gap between clean focal audio and noisy real-world soundscapes.

In [ ]:
# # =================================================================
# # S4 -- DATA PIPELINE
# # =================================================================

# def load_int16(path):
#     """Load int16 waveform tensor to float32 in [-1, 1]."""
#     waveform_int16 = torch.load(path, map_location="cpu")
#     return waveform_int16.float() / 32767.0

# _FC = {}
# def load_focal(p):
#     """Load focal waveform with simple LRU cache."""
#     if p in _FC: return _FC[p]
#     pp = WAVEFORM_CACHE_DIR / p
#     if not pp.exists(): return None
#     a = load_int16(pp).numpy()
#     if len(_FC) >= 2000:
#         _FC.pop(next(iter(_FC)))
#     _FC[p] = a
#     return a

# _SC_CACHE = {}
# def load_sc_waveform_from(cache_dir, cache_file):
#     """Load a soundscape waveform with LRU cache."""
#     key = str(cache_dir / cache_file)
#     if key in _SC_CACHE: return _SC_CACHE[key]
#     pp = cache_dir / cache_file
#     if not pp.exists(): return None
#     a = load_int16(pp).numpy()
#     if len(_SC_CACHE) >= 200:
#         _SC_CACHE.pop(next(iter(_SC_CACHE)))
#     _SC_CACHE[key] = a
#     return a

# def extract_chunk_np(waveform, start_sample, n_samples):
#     """Extract a chunk, left-padding if the recording is too short."""
#     total = len(waveform)
#     if total <= n_samples:
#         return np.pad(waveform, (n_samples - total, 0))
#     end = start_sample + n_samples
#     if end > total:
#         start_sample = max(0, total - n_samples)
#     return waveform[start_sample:start_sample + n_samples]

# def apply_aug(w):
#     """Simple waveform augmentation: gain jitter + noise + shift."""
#     if np.random.random() < AUG_PROB:
#         w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
#     if np.random.random() < AUG_PROB:
#         sp = (w ** 2).mean()
#         if sp > 1e-10:
#             w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
#                 sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
#     return w

# # ------------------------------------------------------------------
# # Build soundscape MixUp pool (labeled windows only)
# # ------------------------------------------------------------------
# sc_mixup_sources = []
# _sc_file_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_file_meta.csv")
# _sc_file_dict = dict(zip(_sc_file_meta["filename"], _sc_file_meta["cache_file"]))
# _labeled_rows = []
# for i in range(len(sc_cache_meta)):
#     row = sc_cache_meta.iloc[i]
#     if Y_SC[i].sum() > 0:
#         cf = _sc_file_dict.get(row["filename"])
#         if cf is not None:
#             _labeled_rows.append({
#                 "filename": row["filename"], "start_sec": int(row["start_sec"]),
#                 "cache_file": cf, "label_idx": i, "fold": int(row.get("fold", -1)),
#             })
# if _labeled_rows:
#     _labeled_meta = pd.DataFrame(_labeled_rows)
#     sc_mixup_sources.append((WAVEFORM_CACHE_DIR, _labeled_meta, Y_SC))
#     print(f"SC MixUp pool: {len(_labeled_meta)} labeled windows")

# # ------------------------------------------------------------------
# # FocalDS -- with Focal-Focal AND Focal-Soundscape MixUp
# # ------------------------------------------------------------------
# class FocalDS(Dataset):
#     """Focal recording dataset. Returns (waveform, label, weight, mask, source_tag)."""
#     def __init__(self, df, l2i, secondary_lookup=None,
#                  sc_mixup_sources=None, fold_k=None, aug=False):
#         self.df, self.l2i, self.aug = df.reset_index(drop=True), l2i, aug
#         self.secondary_lookup = secondary_lookup
#         self.sc_mixup_sources = sc_mixup_sources
#         self.fold_k = fold_k

#     def __len__(self): return len(self.df)

#     def _load_chunk(self, r):
#         w = load_focal(r["cache_file"])
#         if w is None: return None, None
#         if self.aug:
#             start = np.random.randint(0, max(1, len(w) - TRAIN_SAMPLES + 1)) if len(w) > TRAIN_SAMPLES else 0
#         else:
#             start = int(r.get("start_sec", 0)) * SR
#         ch = extract_chunk_np(w, start, TRAIN_SAMPLES)
#         lb = np.zeros(NUM_CLASSES, dtype=np.float32)
#         if str(r["primary_label"]) in self.l2i:
#             lb[self.l2i[str(r["primary_label"])]] = 1.0
#         if self.secondary_lookup is not None and "original_idx" in self.df.columns:
#             for s in self.secondary_lookup.get(int(r["original_idx"]), []):
#                 if s in self.l2i: lb[self.l2i[s]] = 1.0
#         return ch, lb

#     def __getitem__(self, i):
#         r1 = self.df.iloc[i]
#         ch1, lb1 = self._load_chunk(r1)
#         if ch1 is None:
#             return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES),
#                     torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal_missing", torch.tensor(0.0, dtype=torch.float32))

#         # Focal-Focal MixUp
#         if USE_FOCAL_MIXUP and self.aug and np.random.random() < MIXUP_PROB:
#             ch2 = None
#             for _ in range(3):
#                 j = np.random.randint(len(self.df))
#                 ch2, lb2 = self._load_chunk(self.df.iloc[j])
#                 if ch2 is not None: break
#             if ch2 is not None:
#                 lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
#                 ch_mix = (lam * ch1 + (1 - lam) * ch2).astype(np.float32)
#                 if self.aug: ch_mix = apply_aug(ch_mix)
#                 lb = np.maximum(lb1, lb2) if MIXUP_HARD else (lam * lb1 + (1 - lam) * lb2)
#                 return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
#                         torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal", torch.tensor(0.0, dtype=torch.float32))

#         # Focal-Soundscape MixUp
#         if (USE_FOCAL_SC_MIXUP and self.aug and self.sc_mixup_sources
#                 and np.random.random() < FOCAL_SC_MIXUP_PROB):
#             src_idx = np.random.randint(len(self.sc_mixup_sources))
#             cache_dir, meta_df_sc, labels = self.sc_mixup_sources[src_idx]
#             eligible = meta_df_sc[meta_df_sc["fold"] != self.fold_k] if self.fold_k is not None else meta_df_sc
#             if len(eligible) > 0:
#                 sc_row = eligible.iloc[np.random.randint(len(eligible))]
#                 sc_wav = load_sc_waveform_from(cache_dir, sc_row["cache_file"])
#                 if sc_wav is not None and len(sc_wav) >= TRAIN_SAMPLES:
#                     sc_chunk = extract_chunk_np(sc_wav, int(sc_row["start_sec"]) * SR, TRAIN_SAMPLES)
#                     lam = np.random.beta(FOCAL_SC_MIXUP_ALPHA, FOCAL_SC_MIXUP_ALPHA)
#                     ch_mix = (lam * ch1 + (1 - lam) * sc_chunk).astype(np.float32)
#                     if self.aug: ch_mix = apply_aug(ch_mix)
#                     lb_sc = labels[int(sc_row["label_idx"])].astype(np.float32)
#                     lb = np.maximum(lb1, lb_sc) if MIXUP_HARD else lam * lb1 + (1 - lam) * lb_sc
#                     return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
#                             torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal", torch.tensor(0.0, dtype=torch.float32))

#         # No MixUp
#         if self.aug: ch1 = apply_aug(ch1)
#         return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0),
#                 torch.from_numpy(lb1),
#                 torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal", torch.tensor(0.0, dtype=torch.float32))

# # ------------------------------------------------------------------
# # ScDS -- Labeled soundscape windows
# # ------------------------------------------------------------------
# class ScDS(Dataset):
#     def __init__(self, Y, sc_df, aug=False):
#         self.Y, self.df, self.aug = Y, sc_df.reset_index(drop=True), aug
#     def __len__(self): return len(self.Y)
#     def __getitem__(self, i):
#         row = self.df.iloc[i]
#         wav_full = load_sc_waveform_from(WAVEFORM_CACHE_DIR, row.get("cache_file")) \
#                    if row.get("cache_file") else None
#         if wav_full is None:
#             wav_t = torch.zeros(1, TRAIN_SAMPLES)
#         else:
#             chunk = extract_chunk_np(wav_full, int(row["start_sec"]) * SR, TRAIN_SAMPLES)
#             if self.aug: chunk = apply_aug(chunk)
#             wav_t = torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0)
#         return (wav_t, torch.from_numpy(self.Y[i].astype(np.float32)),
#                 torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "sc", torch.tensor(1.0, dtype=torch.float32))

# # ------------------------------------------------------------------
# # ULScDS -- Unlabeled soundscape windows (has_expert_label == False)
# # ------------------------------------------------------------------
# class ULScDS(Dataset):
#     def __init__(self, df, aug=False):
#         self.df = df.reset_index(drop=True)
#         self.aug = aug

#     def __len__(self): return len(self.df)

#     def __getitem__(self, i):
#         row = self.df.iloc[i]
#         wav_full = load_sc_waveform_from(WAVEFORM_CACHE_DIR, row.get("cache_file")) \
#                    if row.get("cache_file") else None

#         if wav_full is None:
#             wav_t = torch.zeros(1, TRAIN_SAMPLES)
#         else:
#             chunk = extract_chunk_np(wav_full, int(row["start_sec"]) * SR, TRAIN_SAMPLES)
#             if self.aug: chunk = apply_aug(chunk)
#             wav_t = torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0)

#         # Ground Truthがないため、0埋めベクトルを返す
#         lb = torch.zeros(NUM_CLASSES, dtype=torch.float32)

#         # is_soundscape フラグとして 2.0 を返す（ulsc識別用）
#         return (wav_t, lb,
#                 torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "ulsc", torch.tensor(2.0, dtype=torch.float32))

# # ------------------------------------------------------------------
# # Load focal secondary labels
# # ------------------------------------------------------------------
# focal_secondary_labels = None
# if USE_FOCAL_SECONDARY:
#     focal_secondary_labels = {}
#     for idx, row in train_df.iterrows():
#         sec = row.get("secondary_labels", "")
#         if pd.isna(sec) or sec in ("", "[]"): continue
#         try:
#             sec_list = eval(sec) if isinstance(sec, str) else []
#         except: continue
#         valid = [s for s in sec_list if s in LABEL2IDX]
#         if valid: focal_secondary_labels[idx] = valid
#     print(f"Focal secondary labels: {len(focal_secondary_labels)} files")

# # ------------------------------------------------------------------
# # Multi-source batch sampler
# # ------------------------------------------------------------------
# class MixSamp(torch.utils.data.Sampler):
#     """Controls batch composition via per-source shares."""
#     def __init__(self, sizes, names, shares, bs, nst, seed=0):
#         self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
#         self.rng = np.random.default_rng(seed)
#         per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
#         total = sum(per_src)
#         if total != bs:
#             per_src[int(np.argmax(per_src))] += (bs - total)
#         self.per_src = per_src
#         self.offsets = [0]
#         for s in sizes[:-1]:
#             self.offsets.append(self.offsets[-1] + s)
#     def __len__(self): return self.nst
#     def __iter__(self):
#         for _ in range(self.nst):
#             batch = []
#             for off, size, n in zip(self.offsets, self.sizes, self.per_src):
#                 if n <= 0 or size <= 0: continue
#                 idxs = self.rng.integers(0, size, size=n)
#                 batch.extend([off + int(i) for i in idxs])
#             self.rng.shuffle(batch)
#             yield batch

# def collate_m(batch):
#     return (torch.stack([b[0] for b in batch]),
#             torch.stack([b[1] for b in batch]),
#             torch.stack([b[2] for b in batch]),
#             torch.stack([b[3] for b in batch]),
#             [b[4] for b in batch],
#             torch.stack([b[5] for b in batch])
#             )

# def mk_sw(sr):
#     """Per-sample source weight tensor."""
#     return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

# print("OK Data pipeline ready")

In [ ]:
# =================================================================
# S4 -- DATA PIPELINE (Chunk-based Memory Optimized)
# =================================================================
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset

# チャンクファイルの保存先ディレクトリ（Google Drive等）を指定してください
WAVEFORM_CACHE_DIR = Path('/content/local_chunk_cache')

# ---- チャンクキャッシュ管理 ----
_CHUNK_CACHE = {}
MAX_CHUNKS_IN_MEMORY = 50 # メモリ容量に合わせて調整（1チャンク約数百MB〜数GB）

def load_from_chunk(chunk_filename, dict_key):
    """
    チャンク(辞書)をキャッシュに読み込み、指定されたキー(ファイル名)のテンソルを返す。
    返り値は float32 に正規化された numpy 配列 [-1, 1]。
    """
    # 1. チャンク自体がキャッシュになければロード
    if chunk_filename not in _CHUNK_CACHE:
        chunk_path = WAVEFORM_CACHE_DIR / chunk_filename
        if not chunk_path.exists():
            # print(f"Chunk not found: {chunk_path}")
            return None

        # メモリ制限を超える場合は一番古いキャッシュを破棄 (LRU)
        if len(_CHUNK_CACHE) >= MAX_CHUNKS_IN_MEMORY:
            _CHUNK_CACHE.pop(next(iter(_CHUNK_CACHE)))

        # チャンク（辞書）をロード
        _CHUNK_CACHE[chunk_filename] = torch.load(chunk_path, map_location="cpu")

    # 2. チャンク辞書から特定のファイル(dict_key)を取り出す
    chunk_data = _CHUNK_CACHE[chunk_filename]
    if dict_key not in chunk_data:
        # print(f"Key {dict_key} not found in {chunk_filename}")
        return None

    waveform_int16 = chunk_data[dict_key]

    # 3. int16 -> float32 に変換して返す
    return (waveform_int16.float() / 32767.0).numpy()


# 既存のコードとの互換性のためのラッパー関数
def load_focal(chunk_filename, dict_key):
    return load_from_chunk(chunk_filename, dict_key)

def load_sc_waveform_from(chunk_filename, dict_key):
    return load_from_chunk(chunk_filename, dict_key)

# ------------------------------------------------------------------

def extract_chunk_np(waveform, start_sample, n_samples):
    """Extract a chunk, left-padding if the recording is too short."""
    total = len(waveform)
    if total <= n_samples:
        return np.pad(waveform, (n_samples - total, 0))
    end = start_sample + n_samples
    if end > total:
        start_sample = max(0, total - n_samples)
    return waveform[start_sample:start_sample + n_samples]

def apply_aug(w):
    """Simple waveform augmentation: gain jitter + noise + shift."""
    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
    return w

# ------------------------------------------------------------------
# Build soundscape MixUp pool (labeled windows only)
# ------------------------------------------------------------------
sc_mixup_sources = []
# ★修正: 更新された updated_soundscape_cache_meta.csv を読み込む想定
# _sc_file_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "updated_soundscape_cache_meta.csv")
_sc_file_meta = pd.read_csv("/content/local_chunk_cache/updated_soundscape_cache_meta.csv")

# filename をキーとして、(chunk_file, dict_key) のタプルを取得できるようにする
_sc_file_dict = {}
for _, row in _sc_file_meta.iterrows():
    if "filename" in row and "chunk_file" in row and "dict_key" in row:
        _sc_file_dict[row["filename"]] = (row["chunk_file"], row["dict_key"])

_labeled_rows = []
for i in range(len(sc_cache_meta)):
    row = sc_cache_meta.iloc[i]
    if Y_SC[i].sum() > 0:
        file_info = _sc_file_dict.get(row["filename"])
        if file_info is not None:
            chunk_file, dict_key = file_info
            _labeled_rows.append({
                "filename": row["filename"],
                "start_sec": int(row["start_sec"]),
                "chunk_file": chunk_file,
                "dict_key": dict_key,
                "label_idx": i,
                "fold": int(row.get("fold", -1)),
            })

if _labeled_rows:
    _labeled_meta = pd.DataFrame(_labeled_rows)
    sc_mixup_sources.append((WAVEFORM_CACHE_DIR, _labeled_meta, Y_SC))
    print(f"SC MixUp pool: {len(_labeled_meta)} labeled windows")

# ------------------------------------------------------------------
# FocalDS -- with Focal-Focal AND Focal-Soundscape MixUp
# ------------------------------------------------------------------
class FocalDS(Dataset):
    """Focal recording dataset. Returns (waveform, label, weight, mask, source_tag)."""
    def __init__(self, df, l2i, secondary_lookup=None,
                 sc_mixup_sources=None, fold_k=None, aug=False):
        self.df, self.l2i, self.aug = df.reset_index(drop=True), l2i, aug
        self.secondary_lookup = secondary_lookup
        self.sc_mixup_sources = sc_mixup_sources
        self.fold_k = fold_k

    def __len__(self): return len(self.df)

    def _load_chunk(self, r):
        # ★修正: cache_fileの代わりに chunk_file と dict_key を使用
        w = load_focal(r.get("chunk_file"), r.get("dict_key"))
        if w is None: return None, None

        if self.aug:
            start = np.random.randint(0, max(1, len(w) - TRAIN_SAMPLES + 1)) if len(w) > TRAIN_SAMPLES else 0
        else:
            start = int(r.get("start_sec", 0)) * SR

        ch = extract_chunk_np(w, start, TRAIN_SAMPLES)
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        if str(r["primary_label"]) in self.l2i:
            lb[self.l2i[str(r["primary_label"])]] = 1.0
        if self.secondary_lookup is not None and "original_idx" in self.df.columns:
            for s in self.secondary_lookup.get(int(r["original_idx"]), []):
                if s in self.l2i: lb[self.l2i[s]] = 1.0
        return ch, lb

    def __getitem__(self, i):
        r1 = self.df.iloc[i]
        ch1, lb1 = self._load_chunk(r1)
        if ch1 is None:
            return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES),
                    torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal_missing", torch.tensor(0.0, dtype=torch.float32))

        # Focal-Focal MixUp
        if USE_FOCAL_MIXUP and self.aug and np.random.random() < MIXUP_PROB:
            ch2 = None
            for _ in range(3):
                j = np.random.randint(len(self.df))
                ch2, lb2 = self._load_chunk(self.df.iloc[j])
                if ch2 is not None: break
            if ch2 is not None:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                ch_mix = (lam * ch1 + (1 - lam) * ch2).astype(np.float32)
                if self.aug: ch_mix = apply_aug(ch_mix)
                lb = np.maximum(lb1, lb2) if MIXUP_HARD else (lam * lb1 + (1 - lam) * lb2)
                return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                        torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal", torch.tensor(0.0, dtype=torch.float32))

        # Focal-Soundscape MixUp
        if (USE_FOCAL_SC_MIXUP and self.aug and self.sc_mixup_sources
                and np.random.random() < FOCAL_SC_MIXUP_PROB):
            src_idx = np.random.randint(len(self.sc_mixup_sources))
            cache_dir, meta_df_sc, labels = self.sc_mixup_sources[src_idx]
            eligible = meta_df_sc[meta_df_sc["fold"] != self.fold_k] if self.fold_k is not None else meta_df_sc

            if len(eligible) > 0:
                sc_row = eligible.iloc[np.random.randint(len(eligible))]
                # ★修正: sc_row から chunk_file と dict_key を取得
                sc_wav = load_sc_waveform_from(sc_row.get("chunk_file"), sc_row.get("dict_key"))

                if sc_wav is not None and len(sc_wav) >= TRAIN_SAMPLES:
                    sc_chunk = extract_chunk_np(sc_wav, int(sc_row["start_sec"]) * SR, TRAIN_SAMPLES)
                    lam = np.random.beta(FOCAL_SC_MIXUP_ALPHA, FOCAL_SC_MIXUP_ALPHA)
                    ch_mix = (lam * ch1 + (1 - lam) * sc_chunk).astype(np.float32)
                    if self.aug: ch_mix = apply_aug(ch_mix)
                    lb_sc = labels[int(sc_row["label_idx"])].astype(np.float32)
                    lb = np.maximum(lb1, lb_sc) if MIXUP_HARD else lam * lb1 + (1 - lam) * lb_sc
                    return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                            torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal", torch.tensor(0.0, dtype=torch.float32))

        # No MixUp
        if self.aug: ch1 = apply_aug(ch1)
        return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(lb1),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal", torch.tensor(0.0, dtype=torch.float32))

# ------------------------------------------------------------------
# ScDS -- Labeled soundscape windows
# ------------------------------------------------------------------
class ScDS(Dataset):
    def __init__(self, Y, sc_df, aug=False):
        self.Y, self.df, self.aug = Y, sc_df.reset_index(drop=True), aug
    def __len__(self): return len(self.Y)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        # ★修正: chunk_file と dict_key を渡す
        if pd.notna(row.get("chunk_file")) and pd.notna(row.get("dict_key")):
            wav_full = load_sc_waveform_from(row["chunk_file"], row["dict_key"])
        else:
            wav_full = None

        if wav_full is None:
            wav_t = torch.zeros(1, TRAIN_SAMPLES)
        else:
            chunk = extract_chunk_np(wav_full, int(row["start_sec"]) * SR, TRAIN_SAMPLES)
            if self.aug: chunk = apply_aug(chunk)
            wav_t = torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0)
        return (wav_t, torch.from_numpy(self.Y[i].astype(np.float32)),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "sc", torch.tensor(1.0, dtype=torch.float32))

# ------------------------------------------------------------------
# ULScDS -- Unlabeled soundscape windows (has_expert_label == False)
# ------------------------------------------------------------------
class ULScDS(Dataset):
    def __init__(self, df, aug=False):
        self.df = df.reset_index(drop=True)
        self.aug = aug

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        # ★修正: chunk_file と dict_key を渡す
        if pd.notna(row.get("chunk_file")) and pd.notna(row.get("dict_key")):
            wav_full = load_sc_waveform_from(row["chunk_file"], row["dict_key"])
        else:
            wav_full = None

        if wav_full is None:
            wav_t = torch.zeros(1, TRAIN_SAMPLES)
        else:
            chunk = extract_chunk_np(wav_full, int(row["start_sec"]) * SR, TRAIN_SAMPLES)
            if self.aug: chunk = apply_aug(chunk)
            wav_t = torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0)

        # Ground Truthがないため、0埋めベクトルを返す
        lb = torch.zeros(NUM_CLASSES, dtype=torch.float32)

        # is_soundscape フラグとして 2.0 を返す（ulsc識別用）
        return (wav_t, lb,
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "ulsc", torch.tensor(2.0, dtype=torch.float32))

# class ULScDS(Dataset):
#     def __init__(self, df, aug=False):
#         self.df = df.reset_index(drop=True)
#         self.aug = aug

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, i):
#         row = self.df.iloc[i]
#         # ★修正: chunk_file と dict_key を渡す
#         if pd.notna(row.get("chunk_file")) and pd.notna(row.get("dict_key")):
#             wav_full = load_sc_waveform_from(row["chunk_file"], row["dict_key"])
#         else:
#             wav_full = None

#         if wav_full is None:
#             wav_t = torch.zeros(1, TRAIN_SAMPLES)
#         else:
#             total_samples = len(wav_full)
#             if total_samples <= TRAIN_SAMPLES:
#                 # 5秒未満の場合はパディング
#                 chunk = np.pad(wav_full, (0, TRAIN_SAMPLES - total_samples))
#             else:
#                 # ★ここがランダムクロップの肝★
#                 # 0 から (全体長 - 5秒) の間でランダムな開始位置を決定
#                 max_start = total_samples - TRAIN_SAMPLES
#                 start_sample = np.random.randint(0, max_start + 1)
#                 chunk = wav_full[start_sample : start_sample + TRAIN_SAMPLES]

#             if self.aug: chunk = apply_aug(chunk)
#             wav_t = torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0)

#         # UlScDSなのでラベルはダミー（後でオンザフライ推論で上書きする）
#         dummy_label = torch.zeros(NUM_CLASSES)

#         # is_soundscape フラグとして 2.0 を返す（ulsc識別用）
#         return (wav_t, dummy_label,
#                 torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "ulsc", torch.tensor(2.0, dtype=torch.float32))

# ------------------------------------------------------------------
# Load focal secondary labels
# ------------------------------------------------------------------
focal_secondary_labels = None
if USE_FOCAL_SECONDARY:
    focal_secondary_labels = {}
    for idx, row in train_df.iterrows():
        sec = row.get("secondary_labels", "")
        if pd.isna(sec) or sec in ("", "[]"): continue
        try:
            sec_list = eval(sec) if isinstance(sec, str) else []
        except: continue
        valid = [s for s in sec_list if s in LABEL2IDX]
        if valid: focal_secondary_labels[idx] = valid
    print(f"Focal secondary labels: {len(focal_secondary_labels)} files")

# ------------------------------------------------------------------
# Multi-source batch sampler
# ------------------------------------------------------------------
class MixSamp(torch.utils.data.Sampler):
    """Controls batch composition via per-source shares."""
    def __init__(self, sizes, names, shares, bs, nst, seed=0):
        self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
        self.rng = np.random.default_rng(seed)
        per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
        total = sum(per_src)
        if total != bs:
            per_src[int(np.argmax(per_src))] += (bs - total)
        self.per_src = per_src
        self.offsets = [0]
        for s in sizes[:-1]:
            self.offsets.append(self.offsets[-1] + s)
    def __len__(self): return self.nst
    def __iter__(self):
        for _ in range(self.nst):
            batch = []
            for off, size, n in zip(self.offsets, self.sizes, self.per_src):
                if n <= 0 or size <= 0: continue
                idxs = self.rng.integers(0, size, size=n)
                batch.extend([off + int(i) for i in idxs])
            self.rng.shuffle(batch)
            yield batch

def collate_m(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]),
            [b[4] for b in batch],
            torch.stack([b[5] for b in batch])
            )

def mk_sw(sr):
    """Per-sample source weight tensor."""
    return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

print("OK Data pipeline ready")

# S5 — Training Loop

* Training recipe¶  
Each epoch:

    1. Sample batches with 90% focal / 10% soundscape composition
    2. Compute mel spectrogram on GPU + apply SpecAugment
    3. Forward pass: backbone produces features for both the distillation head and SED head (on detached features)
    4. Loss = BCE_classification + α × MSE_distillation
    5. Validate on held-out soundscape windows using clip+frame blend
* Perch distillation mechanics  
For each batch, the frozen Perch ONNX model processes the same 5-second waveforms to produce target embeddings. The student’s distillation head projects GAP’d backbone features to 1536-d and is trained to match Perch via MSE. Since Perch trained on 14,795 species with millions of hours, this distillation teaches the small EfficientNet-B0 backbone to produce rich, generalizable audio representations.

* Learning rate schedule  
Linear warmup over 2 epochs, then cosine decay to 1e-6. Gradient clipping at norm 1.0.

In [ ]:
model_teacher_path = "/content/drive/MyDrive/kaggle/Birdclef2026/outputs/exp27/"
# model_teacher_path = "/content/drive/MyDrive/kaggle/Birdclef2026/outputs/exp17/"
MODELS = [model_teacher_path + f'souped_fold{i}_best_macro_auc.pt' for i in range(5)]
models_teacher = []
for path in MODELS:
    model_teacher = make_model(MODEL_NAME="regnety_008.pycls_in1k")
    # model_teacher = make_model(MODEL_NAME="seresnext26t_32x4d")
    model_teacher.load_state_dict(torch.load(path), strict=False)

    models_teacher.append(model_teacher.to(device))

In [ ]:
# =================================================================
# S5 -- TRAINING --- 5位のpseudo labeling (secondary labelの付与) ---
# =================================================================

def _load_val_waveforms(val_sc_df):
    """Load validation waveforms (always 5s)."""
    # sc_file_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_file_meta.csv")
    sc_file_meta = pd.read_csv("/content/local_chunk_cache/updated_soundscape_cache_meta.csv")
    # sc_file_dict = dict(zip(sc_file_meta["filename"], sc_file_meta["cache_file"]))

    # 変更点: filename をキーとして、(chunk_file, dict_key) のタプルを取得する辞書を作成
    sc_file_dict = {}
    for _, row in sc_file_meta.iterrows():
        if "filename" in row and "chunk_file" in row and "dict_key" in row:
            sc_file_dict[row["filename"]] = (row["chunk_file"], row["dict_key"])

    # wavs = []
    # for _, row in val_sc_df.iterrows():
    #     cf = sc_file_dict.get(row["filename"])
    #     if cf is not None:
    #         w = load_sc_waveform_from(WAVEFORM_CACHE_DIR, cf)
    #         if w is not None:
    #             chunk = extract_chunk_np(w, int(row["start_sec"]) * SR, VAL_SAMPLES)
    #             wavs.append(torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0))
    #         else: wavs.append(torch.zeros(1, VAL_SAMPLES))
    #     else: wavs.append(torch.zeros(1, VAL_SAMPLES))

    wavs = []
    for _, row in val_sc_df.iterrows():
        file_info = sc_file_dict.get(row["filename"])

        if file_info is not None:
            chunk_file, dict_key = file_info

            # 変更点: load_sc_waveform_from に chunk_file と dict_key を渡す
            if pd.notna(chunk_file) and pd.notna(dict_key):
                w = load_sc_waveform_from(chunk_file, dict_key)
            else:
                w = None

            if w is not None:
                chunk = extract_chunk_np(w, int(row["start_sec"]) * SR, VAL_SAMPLES)
                wavs.append(torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0))
            else:
                wavs.append(torch.zeros(1, VAL_SAMPLES))
        else:
            wavs.append(torch.zeros(1, VAL_SAMPLES))

    return wavs

def _predict_from_waveforms(model, mel_transform, wav_list, batch_size=64):
    """Inference: mel -> model -> sigmoid. Distillation head is NOT used."""
    model.eval()
    preds_clip, preds_fmax, preds_blend = [], [], []
    with torch.no_grad():
        for s in range(0, len(wav_list), batch_size):
            batch = torch.stack(wav_list[s:s+batch_size]).to(device)
            mel = mel_transform(batch)
            B = mel.size(0)
            for i in range(B):
                mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            with autocast():
                clip_logits, framewise = model(mel, return_framewise=True)
                frame_max = framewise.max(dim=1).values
                p_clip = torch.sigmoid(clip_logits).cpu().numpy()
                p_fmax = torch.sigmoid(frame_max).cpu().numpy()
                p_blend = 0.5 * p_clip + 0.5 * p_fmax
            preds_clip.append(p_clip); preds_fmax.append(p_fmax); preds_blend.append(p_blend)
    return {"clip": np.concatenate(preds_clip), "fmax": np.concatenate(preds_fmax),
            "blend": np.concatenate(preds_blend)}

def build_active_datasets(fold_k, epoch=0):
    items = []
    if USE_FOCAL:
        fds = FocalDS(audio_cache_meta[audio_cache_meta["fold"] != fold_k],
                      LABEL2IDX, secondary_lookup=focal_secondary_labels,
                      sc_mixup_sources=sc_mixup_sources if USE_FOCAL_SC_MIXUP else None,
                      fold_k=fold_k, aug=True)
        items.append(("focal", fds, len(fds)))
    if USE_LABELED_SC:
        vm = sc_cache_meta["fold"].values == fold_k
        sc_train_df = sc_cache_meta[~vm].reset_index(drop=True)
        Y_tr = Y_SC[~vm]
        sds = ScDS(Y_tr, sc_train_df, aug=True)
        items.append(("sc", sds, len(sds)))

    # --- ここから追加 ---
    if USE_UNLABELED_SC:
        # メタデータを読み込み、Expert Labelがないものを抽出
        # ulsc_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "pseudo_window_meta.csv")
        ulsc_meta = pd.read_csv("/content/local_chunk_cache/updated_pseudo_window_meta.csv")
        ulsc_df = ulsc_meta[ulsc_meta["has_expert_label"] == False].reset_index(drop=True)

        # Fold情報のマッピング (ValidationへのLeak防止)
        file_to_fold = dict(zip(sc_cache_meta["filename"], sc_cache_meta["fold"]))
        ulsc_df["fold"] = ulsc_df["filename"].map(file_to_fold).fillna(-1).astype(int)

        vm_ulsc = ulsc_df["fold"].values == fold_k
        ulsc_train_df = ulsc_df[~vm_ulsc].reset_index(drop=True)
        current_seed = 42 + epoch
        ulsc_train_df = ulsc_train_df.sample(frac=0.1, random_state=current_seed).reset_index(drop=True) # random_state=42

        uds = ULScDS(ulsc_train_df, aug=True)
        items.append(("ulsc", uds, len(uds)))
    # --- ここまで追加 ---

    # --- 確認用コードをリターン前に追加 ---
    if epoch == 0:
        print(f"\n[Fold {fold_k}] Active Dataset Sizes:")
        for name, ds, length in items:
            print(f"  - {name} ({type(ds).__name__}): {length} 件")

    return items

def train_fold(fold_k):
    vm = sc_cache_meta["fold"].values == fold_k
    Y_val = Y_SC[vm]
    ns22_val = non_s22_mask_sc[vm]
    val_sc_df = sc_cache_meta[vm].reset_index(drop=True)

    active = build_active_datasets(fold_k)
    names, datasets, sizes = zip(*active)
    mds = ConcatDataset(list(datasets))
    nst = max(100, int(sum(sizes) / BATCH))

    print(f"  Streams: {dict(zip(names, sizes))}  steps/ep: {nst}")

    m = make_model()
    mel_transform = MelSpecTransform().to(device)
    spec_augment = SpecAugment().to(device)
    perch_teacher = PerchTeacher(PERCH_ONNX_PATH,
                                  "cuda" if torch.cuda.is_available() else "cpu") \
                    if USE_PERCH_DISTILL else None

    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    scaler = GradScaler()
    warmup_steps = nst * WARMUP_EPOCHS
    total_steps  = nst * EPOCHS
    warmup_sched = torch.optim.lr_scheduler.LinearLR(opt, start_factor=1/25, end_factor=1.0,
                                                      total_iters=warmup_steps)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps - warmup_steps,
                                                               eta_min=1e-6)
    sch = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_steps])

    history = {"ep": [], "train_loss": [], "cls_loss": [], "dist_loss": [],
               "macro": [], "ns22_macro": [],
               "ns22_Aves": [], "ns22_Amphibia": [], "ns22_Insecta": [], "ns22_Mammalia": [],
               "val_preds": []}
    best_ns22, best_state_ns22 = -1.0, None
    best_macro, best_state_macro = -1.0, None
    val_wavs = _load_val_waveforms(val_sc_df)

    # Early stopping parameters
    patience = 5
    early_stop_counter = 0

    # 全エポックのスコアと重みを保存するための空リストを用意
    all_states = []

    for ep in range(EPOCHS):
        m.train()
        active = build_active_datasets(fold_k, epoch=ep)
        names, datasets, sizes = zip(*active)
        mds = ConcatDataset(list(datasets))
        smp = MixSamp(list(sizes), list(names), SHARES, BATCH, nst, seed=42 + ep)
        tl = DataLoader(mds, batch_sampler=smp, collate_fn=collate_m,
                        num_workers=0, pin_memory=True)
        el, el_cls, el_dist, nb_count = 0.0, 0.0, 0.0, 0
        t0 = time.time()

        for wav, lb, wt, mk, sr, is_soundscape in tl:
            wav, lb, wt, mk, is_soundscape = wav.to(device), lb.to(device), wt.to(device), mk.to(device), is_soundscape.to(device).unsqueeze(1)
            sw = mk_sw(sr).to(device)
            base_labels = lb

            with torch.no_grad():
                # ---------------------------------------------------
                # 1. クリーンなMelSpectrogramの生成 (Teacher用)
                # ---------------------------------------------------
                mel_clean = mel_transform(wav)
                B = mel_clean.size(0)
                for i in range(B):
                    mel_clean[i] = (mel_clean[i] - mel_clean[i].mean()) / (mel_clean[i].std() + 1e-6)

                # ※ここではまだ SpecAugment はかけません。Teacherは綺麗な波形で推論します。

            if USE_PSEUDO_LABEL:
                # ---------------------------------------------------
                # 2. 教師モデルによるオンザフライ推論
                # ---------------------------------------------------
                pseudo_prob_max = None
                pseudo_framemax_prob_max = None

                for model_teacher in models_teacher:
                    with torch.no_grad():
                        model_teacher.eval()
                        # Teacherには augmentation の無い mel_clean を渡す
                        current_teacher_logits, framewise = model_teacher(mel_clean, return_framewise=True)
                        current_teacher_framemax = framewise.max(dim=1).values

                        current_prob = torch.sigmoid(current_teacher_logits)
                        current_framemax_prob = torch.sigmoid(current_teacher_framemax)

                        # 【1位解法適用】平均(Mean)ではなく、最大値(Max)でアンサンブルしてコントラストを維持
                        if pseudo_prob_max is None:
                            pseudo_prob_max = current_prob
                            pseudo_framemax_prob_max = current_framemax_prob
                        else:
                            pseudo_prob_max = torch.max(pseudo_prob_max, current_prob)
                            pseudo_framemax_prob_max = torch.max(pseudo_framemax_prob_max, current_framemax_prob)

                # ブレンド
                pseudo_labels = pseudo_prob_max * 0.5 + pseudo_framemax_prob_max * 0.5

                # ---------------------------------------------------
                # 3. 【1位解法適用】 べき乗変換 (Power Transform)
                # ---------------------------------------------------
                # 1.0より大きいべき乗(gamma)をかけることで、ノイズとなる低い確率をゼロに近づけ、
                # 確信度の高い推論だけを鋭く残す (Sharpening)
                gamma = 1.6
                pseudo_labels = pseudo_labels ** gamma

                # ---------------------------------------------------
                # 4. ラベルの合成 (Hard GTの保持)
                # ---------------------------------------------------
                # Focal / Labeled Sc用: 専門家ラベルの 1.0 を薄めないよう max を取る
                labeled_target = torch.max(base_labels, pseudo_labels)

                # Unlabeled Sc用: べき乗変換でノイズ除去済みのため、ハードな閾値は外す
                ulsc_labels = pseudo_labels

                # マスクによる適用
                is_focal_mask = (is_soundscape == 0.0).float()
                is_sc_mask = (is_soundscape == 1.0).float()
                is_ulsc_mask = (is_soundscape == 2.0).float()

                final_labels = (is_focal_mask * labeled_target +
                                is_sc_mask * labeled_target +
                                is_ulsc_mask * ulsc_labels)
            else:
                final_labels = base_labels

            # ---------------------------------------------------
            # 5. 【1位解法適用】 Student用のノイズ付与 (MixUp & SpecAugment)
            # ---------------------------------------------------
            mel_student = mel_clean.clone()

            # バッチ内MixUp: FocalとUnlabeled Scを混ぜ合わせ、強力なデータ拡張を行う
            if np.random.rand() < 0.5:
                lam = np.random.beta(0.5, 0.5)
                rand_index = torch.randperm(mel_student.size(0)).to(device)

                # 波形(Mel)のブレンド
                mel_student = lam * mel_student + (1 - lam) * mel_student[rand_index, :]
                # ラベルのブレンド
                final_labels = lam * final_labels + (1 - lam) * final_labels[rand_index, :]
                # Loss計算用のウェイトとマスクのブレンド
                wt = lam * wt + (1 - lam) * wt[rand_index]
                mk = lam * mk + (1 - lam) * mk[rand_index, :]

            # 最後に Student用の Mel に対して SpecAugment を適用
            mel_student = spec_augment(mel_student)

            # ---------------------------------------------------
            # 6. Studentモデルの学習
            # ---------------------------------------------------
            with autocast():
                if USE_PERCH_DISTILL:
                    # Studentモデルには強いノイズ(mel_student)を入力
                    clip_logits, framewise, distill_emb = m(mel_student, return_framewise=True, return_distill=True)
                else:
                    clip_logits, framewise = m(mel_student, return_framewise=True)

                frame_max_logits = framewise.max(dim=1).values

                # BCE loss using the blended/pseudo labels (final_labels) as targets
                bce_clip_final = F.binary_cross_entropy_with_logits(clip_logits, final_labels, reduction="none")
                bce_frame_final = F.binary_cross_entropy_with_logits(frame_max_logits, final_labels, reduction="none")

                # Combine clip and frame BCE losses per sample and per class
                loss_sed_per_sample_per_class = (0.5 * bce_clip_final + 0.5 * bce_frame_final)

                # Apply sample weights and mask to the loss
                ps = (loss_sed_per_sample_per_class * wt * mk).sum(1) / (mk.sum(1) + 1e-8)
                cls_loss = (ps * sw).mean()

                # Distillation loss
                if USE_PERCH_DISTILL and perch_teacher is not None:
                    with torch.no_grad():
                        wav_5s = wav.squeeze(1)
                        N = wav_5s.shape[1]
                        if N > 160000:
                            start = (N - 160000) // 2
                            wav_5s = wav_5s[:, start:start + 160000]
                        elif N < 160000:
                            wav_5s = F.pad(wav_5s, (0, 160000 - N))
                        perch_emb = perch_teacher.embed(wav_5s).to(device)
                    distill_loss = 1.0 - F.cosine_similarity(distill_emb, perch_emb, dim=-1).mean()
                    loss = cls_loss + ALPHA_DISTILL * distill_loss
                else:
                    distill_loss = torch.tensor(0.0)
                    loss = cls_loss

            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()
            el += loss.item(); el_cls += cls_loss.item()
            el_dist += distill_loss.item(); nb_count += 1

        # Validation
        val_preds_dict = _predict_from_waveforms(m, mel_transform, val_wavs)
        val_preds = val_preds_dict["blend"]
        r = full_eval(Y_val, val_preds, ns22_val, TAXON_MASKS)
        for mode in ["clip", "fmax", "blend"]:
            r_mode = full_eval(Y_val, val_preds_dict[mode], ns22_val, TAXON_MASKS)
            r[f"ns22_{mode}"] = r_mode["non_s22_macro"]

        history["ep"].append(ep)
        history["train_loss"].append(round(el / nb_count, 5))
        history["cls_loss"].append(round(el_cls / nb_count, 5))
        history["dist_loss"].append(round(el_dist / nb_count, 5))
        history["macro"].append(r["macro_auc_all"])
        history["ns22_macro"].append(r["non_s22_macro"])
        for t in ["Aves", "Amphibia", "Insecta", "Mammalia"]:
            history[f"ns22_{t}"].append(r[f"non_s22_{t}"])
        history["val_preds"].append(val_preds.astype(np.float32))

        tag_ns22 = ""; tag_macro = ""

        current_macro = r["macro_auc_all"]
        # --- 2. モデル状態の取得と保存 ---
        # ☢☢重要: 毎エポックの重みをそのまま保存するとGPUメモリ(VRAM)が溢れるため、
        # 必ず .cpu().clone() を使ってCPUのシステムメモリ上に退避させます。
        current_state = {k: v.cpu().clone() for k, v in m.state_dict().items()}

        # リストに (スコア, 重み) のタプルを追加
        all_states.append((current_macro, current_state))

        if r["non_s22_macro"] > best_ns22:
            best_ns22 = r["non_s22_macro"]
            best_state_ns22 = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_ns22 = " *ns22"

        if r["macro_auc_all"] > best_macro:
            best_macro = r["macro_auc_all"]
            best_state_macro = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_macro = " *macro"
            early_stop_counter = 0 # Reset early stopping counter
        else:
            early_stop_counter += 1 # Increment early stopping counter

        dist_str = f" dist={el_dist/nb_count:.4f}" if USE_PERCH_DISTILL else ""
        print(f"    Ep{ep:02d}: loss={el/nb_count:.4f} cls={el_cls/nb_count:.4f}{dist_str} "
              f"lr={opt.param_groups[0]['lr']:.1e} | "
              f"ns22: {r['ns22_blend']:.4f} | "
              f"macro: {r['macro_auc_all']:.4f} | "
              f"Av={r['non_s22_Aves']:.4f} Am={r['non_s22_Amphibia']:.4f} "
              f"In={r['non_s22_Insecta']:.4f} Ma={r['non_s22_Mammalia']:.4f} "
              f"[{time.time()-t0:.0f}s]{tag_ns22}{tag_macro}")

        # Check for early stopping
        if early_stop_counter >= patience:
            print(f"    Early stopping triggered after {patience} epochs without improvement.")
            break

    print(f"--- Starting Greedy Soup based on macro_auc_all ---")

    # 1. 保存した全エポックの状態をスコア順にソート（all_statesに各エポックの情報を保存しておいたと仮定）
    all_states.sort(key=lambda x: x[0], reverse=True)

    # 実行時間節約のため上位K個に絞る
    TOP_K = 5
    all_states = all_states[:TOP_K]

    # 2. 1位のモデルで初期化
    best_soup_score, best_soup_state = all_states[0]
    current_soup_state = {k: v.clone() for k, v in best_soup_state.items()}
    num_models_in_soup = 1

    # --- Initialize a temporary model for souping evaluation ---
    temp_model_for_soup = make_model()

    # 3. 2位以降のモデルを順番にテスト
    for i in range(1, len(all_states)):
        candidate_score, candidate_state = all_states[i]

        # --- 一時的なSoupの作成 (Moving Average) ---
        temp_soup_state = {}
        for key, value in current_soup_state.items():
            if value.dtype.is_floating_point:
                # (今の総和 + 新しい候補) / (今の数 + 1)
                temp_soup_state[key] = (value * num_models_in_soup + candidate_state[key]) / (num_models_in_soup + 1)
            else:
                # BatchNormの追跡変数(int)などはベースモデルのものを維持
                temp_soup_state[key] = value.clone()

        # --- モデルに一時的な重みをロード ---
        temp_model_for_soup.load_state_dict(temp_soup_state)

        # --- ‼‼‼ここでValidationを再実行‼‼‼ ---
        # ノートブック内のバリデーション処理（予測とAUC計算）を再利用し、スコアを取得します。
        soup_val_preds_dict = _predict_from_waveforms(temp_model_for_soup, mel_transform, val_wavs)
        soup_val_preds = soup_val_preds_dict["blend"]
        soup_eval_results = full_eval(Y_val, soup_val_preds, ns22_val, TAXON_MASKS)
        temp_macro = soup_eval_results['macro_auc_all']

        # --- 採用・不採用の判定 ---
        if temp_macro >= best_soup_score:
            print(f"  Model {i} (Epoch score: {candidate_score:.4f}) added! New macro AUC: {temp_macro:.4f}")
            best_soup_score = temp_macro
            current_soup_state = temp_soup_state
            num_models_in_soup += 1
        else:
            print(f"  Model {i} rejected. Temp macro AUC: {temp_macro:.4f}")

    del perch_teacher, m, mel_transform, spec_augment, temp_model_for_soup
    torch.cuda.empty_cache(); gc.collect()
    return best_state_ns22, best_state_macro, history, current_soup_state


In [ ]:
import onnxruntime as ort

# =================================================================
# S6 -- FOLD LOOP + ONNX EXPORT
# =================================================================

if MODE != "train":
    print("Skipping training (MODE='infer')")
    oof_ns22 = None
    all_hist = {}
else:

    oof_ns22 = np.full((len(sc_cache_meta), NUM_CLASSES), np.nan, dtype=np.float32)
    all_hist = {}
    for fold_k in FOLDS:
        print(f"\n{'='*60}")
        print(f"FOLD {fold_k}")
        print(f"{'='*60}")
        vm = sc_cache_meta["fold"].values == fold_k
        val_sc_df_k = sc_cache_meta[vm].reset_index(drop=True)

        best_ns22_state, best_macro_state, hist, current_soup_state = train_fold(fold_k)
        all_hist[fold_k] = hist

        mel_tf = MelSpecTransform().to(device)
        val_wavs_k = _load_val_waveforms(val_sc_df_k)

        if best_macro_state is not None:
            # Save PyTorch checkpoint
            torch.save(best_macro_state, OUT_DIR / f"{EXP}"/ f"fold{fold_k}_best_macro_auc.pt")
            torch.save(current_soup_state, OUT_DIR / f"{EXP}"/ f"souped_fold{fold_k}_best_macro_auc.pt")
            m = make_model()
            m.load_state_dict(best_macro_state, strict=False)
            oof_ns22[vm] = _predict_from_waveforms(m, mel_tf, val_wavs_k)["blend"]

            # --- ONNX Export (Conv1d remap for stable tracing) ---
            m.eval()
            INF_N_MELS = 256
            INF_N_FRAMES = VAL_SAMPLES // HOP_LENGTH + 1
            # INF_N_FRAMES = TRAIN_SAMPLES // HOP_LENGTH + 1

            class SEDExportWrapper(nn.Module):
                def __init__(self, backbone_name, num_classes, backbone_dim, hidden_dim=512):
                    super().__init__()
                    self.backbone = timm.create_model(
                        backbone_name, pretrained=False, in_chans=1,
                        num_classes=0, global_pool="", drop_path_rate=0.15,
                    )
                    self.gem_freq = GeMFreqPool(p_init=3.0)
                    self.dense_drop1 = nn.Dropout(0.25)
                    self.dense_conv = nn.Conv1d(backbone_dim, hidden_dim, 1)
                    self.dense_relu = nn.ReLU(inplace=True)
                    self.dense_drop2 = nn.Dropout(0.5)
                    self.att = nn.Conv1d(hidden_dim, num_classes, 1)
                    self.cla = nn.Conv1d(hidden_dim, num_classes, 1)

                def forward(self, mel):
                    h = self.backbone(mel)
                    h = self.gem_freq(h)
                    h = self.dense_drop1(h)
                    h = self.dense_conv(h)
                    h = self.dense_relu(h)
                    h = self.dense_drop2(h)
                    norm_att = torch.softmax(torch.tanh(self.att(h)), dim=-1)
                    framewise = self.cla(h)
                    clip = torch.sum(norm_att * framewise, dim=2)
                    return clip, framewise.permute(0, 2, 1)

            def load_and_remap_state(export_model, trained_state):
                remap = {}
                for k, v in trained_state.items():
                    if k.startswith("distill_head."):
                        continue
                    if k == "dense.1.weight":
                        remap["dense_conv.weight"] = v.unsqueeze(-1)
                    elif k == "dense.1.bias":
                        remap["dense_conv.bias"] = v
                    else:
                        remap[k] = v
                export_model.load_state_dict(remap, strict=False)

            export_model = SEDExportWrapper(
                BACKBONE_NAME, NUM_CLASSES, m.backbone_dim
            ).to(device)
            # Original model
            load_and_remap_state(export_model, best_macro_state)
            export_model.eval()

            dummy_mel = torch.randn(1, 1, INF_N_MELS, INF_N_FRAMES).to(device)
            onnx_path = OUT_DIR / f"{EXP}"/ f"stage1_sed_distill_fold{fold_k}.onnx"
            torch.onnx.export(
                export_model, dummy_mel, str(onnx_path),
                input_names=["mel"],
                output_names=["clip_logits", "framewise_logits"],
                dynamic_axes={"mel": {0: "batch"},
                              "clip_logits": {0: "batch"},
                              "framewise_logits": {0: "batch"}},
                opset_version=18,
            )

            # Souped model
            load_and_remap_state(export_model, current_soup_state)
            export_model.eval()

            dummy_mel = torch.randn(1, 1, INF_N_MELS, INF_N_FRAMES).to(device)
            onnx_path = OUT_DIR / f"{EXP}"/ f"stage1_souped_sed_distill_fold{fold_k}.onnx"
            torch.onnx.export(
                export_model, dummy_mel, str(onnx_path),
                input_names=["mel"],
                output_names=["clip_logits", "framewise_logits"],
                dynamic_axes={"mel": {0: "batch"},
                              "clip_logits": {0: "batch"},
                              "framewise_logits": {0: "batch"}},
                opset_version=18,
            )

            # Verify
            _sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
            _onnx_out = _sess.run(None, {'mel': dummy_mel.cpu().numpy()})
            with torch.no_grad():
                _ref_clip, _ref_frame = export_model(dummy_mel)
            _diff = np.abs(_ref_clip.cpu().numpy() - _onnx_out[0]).max()
            print(f"  ONNX verify: max|diff|={_diff:.3e}")
            # assert _diff < 2e-3, f"ONNX export diverged: {_diff}" # Increased tolerance
            del _sess

            size_mb = onnx_path.stat().st_size / 1e6
            print(f"  Exported {onnx_path.name} ({size_mb:.1f} MB)")
            del m, export_model


In [ ]:
# [Fold 0] Active Dataset Sizes:
#   - focal (FocalDS): 28906 件
#   - sc (ScDS): 584 件
#   - ulsc (ULScDS): 12714 件
#     Ep00: loss=0.8214 cls=0.1988 dist=0.6226 lr=2.6e-04 | ns22: 0.7177 | macro: 0.7887 | Av=0.7564 Am=0.6811 In=0.7697 Ma=0.2583 [511s] *ns22 *macro
#     Ep01: loss=0.4478 cls=0.0413 dist=0.4065 lr=5.0e-04 | ns22: 0.8542 | macro: 0.8778 | Av=0.9931 Am=0.7855 In=0.8358 Ma=1.0000 [176s] *ns22 *macro
#     Ep02: loss=0.3548 cls=0.0327 dist=0.3221 lr=5.0e-04 | ns22: 0.9046 | macro: 0.9149 | Av=0.9920 Am=0.8599 In=0.8967 Ma=1.0000 [176s] *ns22 *macro
#     Ep03: loss=0.3103 cls=0.0291 dist=0.2812 lr=4.9e-04 | ns22: 0.9178 | macro: 0.9156 | Av=0.9938 Am=0.8731 In=0.9135 Ma=1.0000 [180s] *ns22 *macro
#     Ep04: loss=0.2844 cls=0.0272 dist=0.2572 lr=4.8e-04 | ns22: 0.9183 | macro: 0.9123 | Av=0.9930 Am=0.8790 In=0.9127 Ma=1.0000 [178s] *ns22
#     Ep05: loss=0.2681 cls=0.0265 dist=0.2416 lr=4.6e-04 | ns22: 0.9309 | macro: 0.9175 | Av=0.9923 Am=0.8836 In=0.9385 Ma=1.0000 [188s] *ns22 *macro
#     Ep06: loss=0.2544 cls=0.0258 dist=0.2286 lr=4.4e-04 | ns22: 0.9251 | macro: 0.9172 | Av=0.9786 Am=0.8938 In=0.9265 Ma=1.0000 [182s]
#     Ep07: loss=0.2443 cls=0.0255 dist=0.2188 lr=4.2e-04 | ns22: 0.9064 | macro: 0.9122 | Av=0.9909 Am=0.8944 In=0.8824 Ma=1.0000 [177s]
#     Ep08: loss=0.2359 cls=0.0252 dist=0.2107 lr=3.9e-04 | ns22: 0.9302 | macro: 0.9266 | Av=0.9930 Am=0.9197 In=0.9128 Ma=1.0000 [181s] *macro
#     Ep09: loss=0.2286 cls=0.0248 dist=0.2038 lr=3.7e-04 | ns22: 0.8824 | macro: 0.9136 | Av=0.9923 Am=0.9042 In=0.8264 Ma=1.0000 [181s]
#     Ep10: loss=0.2225 cls=0.0245 dist=0.1980 lr=3.3e-04 | ns22: 0.9154 | macro: 0.9247 | Av=0.9973 Am=0.9155 In=0.8855 Ma=1.0000 [184s]
#     Ep11: loss=0.2163 cls=0.0244 dist=0.1919 lr=3.0e-04 | ns22: 0.9039 | macro: 0.9205 | Av=0.9952 Am=0.9123 In=0.8622 Ma=1.0000 [177s]
#     Ep12: loss=0.2116 cls=0.0243 dist=0.1873 lr=2.7e-04 | ns22: 0.9166 | macro: 0.9241 | Av=0.9966 Am=0.9101 In=0.8907 Ma=1.0000 [178s]
#     Ep13: loss=0.2075 cls=0.0242 dist=0.1833 lr=2.3e-04 | ns22: 0.9067 | macro: 0.9268 | Av=0.9966 Am=0.9195 In=0.8625 Ma=1.0000 [181s] *macro
#     Ep14: loss=0.2030 cls=0.0239 dist=0.1791 lr=2.0e-04 | ns22: 0.9056 | macro: 0.9204 | Av=0.9966 Am=0.9034 In=0.8713 Ma=1.0000 [176s]
#     Ep15: loss=0.1994 cls=0.0238 dist=0.1756 lr=1.7e-04 | ns22: 0.9056 | macro: 0.9208 | Av=0.9966 Am=0.9075 In=0.8695 Ma=1.0000 [181s]
#     Ep16: loss=0.1968 cls=0.0239 dist=0.1729 lr=1.4e-04 | ns22: 0.9011 | macro: 0.9211 | Av=0.9952 Am=0.9035 In=0.8656 Ma=1.0000 [185s]
#     Ep17: loss=0.1933 cls=0.0238 dist=0.1695 lr=1.1e-04 | ns22: 0.9079 | macro: 0.9288 | Av=0.9966 Am=0.9151 In=0.8700 Ma=1.0000 [183s] *macro
#     Ep18: loss=0.1913 cls=0.0236 dist=0.1677 lr=8.0e-05 | ns22: 0.9030 | macro: 0.9233 | Av=0.9966 Am=0.9081 In=0.8625 Ma=1.0000 [174s]
#     Ep19: loss=0.1884 cls=0.0236 dist=0.1649 lr=5.7e-05 | ns22: 0.9063 | macro: 0.9256 | Av=0.9973 Am=0.9148 In=0.8659 Ma=1.0000 [177s]
#     Ep20: loss=0.1874 cls=0.0237 dist=0.1637 lr=3.7e-05 | ns22: 0.9030 | macro: 0.9256 | Av=0.9966 Am=0.9119 In=0.8588 Ma=1.0000 [183s]
#     Ep21: loss=0.1862 cls=0.0236 dist=0.1626 lr=2.2e-05 | ns22: 0.9032 | macro: 0.9253 | Av=0.9966 Am=0.9145 In=0.8568 Ma=1.0000 [174s]
#     Ep22: loss=0.1849 cls=0.0236 dist=0.1613 lr=1.0e-05 | ns22: 0.8996 | macro: 0.9239 | Av=0.9966 Am=0.9098 In=0.8531 Ma=1.0000 [184s]
#     Early stopping triggered after 5 epochs without improvement.
# --- Starting Greedy Soup based on macro_auc_all ---
# V2 backbone: (1, 768, 8, 10)  (C=768)
#   Model 1 rejected. Temp macro AUC: 0.9270
#   Model 2 rejected. Temp macro AUC: 0.9275
#   Model 3 rejected. Temp macro AUC: 0.9274
#   Model 4 rejected. Temp macro AUC: 0.9275

In [ ]:

# ============================================================
# FOLD 1
# ============================================================

# [Fold 1] Active Dataset Sizes:
#   - focal (FocalDS): 28908 件
#   - sc (ScDS): 602 件
#   - ulsc (ULScDS): 12714 件
#   Streams: {'focal': 28908, 'sc': 602, 'ulsc': 12714}  steps/ep: 659
# V2 backbone: (1, 768, 8, 10)  (C=768)

# [Fold 1] Active Dataset Sizes:
#   - focal (FocalDS): 28908 件
#   - sc (ScDS): 602 件
#   - ulsc (ULScDS): 12714 件
#     Ep00: loss=0.2524 cls=0.2524 lr=1.2e-04 | ns22: 0.5341 | macro: 0.7576 | Av=0.8674 Am=0.2628 In=0.4358 Ma=0.2826 [417s] *ns22 *macro
#     Ep01: loss=0.0593 cls=0.0593 lr=2.1e-04 | ns22: 0.6870 | macro: 0.8985 | Av=0.7679 Am=0.8625 In=0.6274 Ma=0.6614 [114s] *ns22 *macro
#     Ep02: loss=0.0521 cls=0.0521 lr=3.1e-04 | ns22: 0.7810 | macro: 0.9244 | Av=0.6707 Am=0.9750 In=0.7828 Ma=0.9315 [115s] *ns22 *macro
#     Ep03: loss=0.0488 cls=0.0488 lr=4.0e-04 | ns22: 0.8613 | macro: 0.9473 | Av=0.8203 Am=0.9750 In=0.8546 Ma=0.9188 [115s] *ns22 *macro
#     Ep04: loss=0.0460 cls=0.0460 lr=5.0e-04 | ns22: 0.9202 | macro: 0.9617 | Av=0.9033 Am=0.9625 In=0.9845 Ma=0.7051 [115s] *ns22 *macro
#     Ep05: loss=0.0433 cls=0.0433 lr=5.0e-04 | ns22: 0.9326 | macro: 0.9665 | Av=0.9300 Am=0.9750 In=0.9781 Ma=0.7692 [114s] *ns22 *macro
#     Ep06: loss=0.0410 cls=0.0410 lr=4.9e-04 | ns22: 0.9339 | macro: 0.9650 | Av=0.8585 Am=0.9625 In=0.9860 Ma=1.0000 [115s] *ns22
#     Ep07: loss=0.0394 cls=0.0394 lr=4.7e-04 | ns22: 0.9545 | macro: 0.9698 | Av=0.9113 Am=0.9750 In=0.9840 Ma=1.0000 [114s] *ns22 *macro
#     Ep08: loss=0.0380 cls=0.0380 lr=4.5e-04 | ns22: 0.9490 | macro: 0.9744 | Av=0.9344 Am=0.9000 In=0.9819 Ma=1.0000 [114s] *macro
#     Ep09: loss=0.0371 cls=0.0371 lr=4.3e-04 | ns22: 0.9509 | macro: 0.9745 | Av=0.9124 Am=0.9750 In=0.9821 Ma=1.0000 [114s] *macro
#     Ep10: loss=0.0361 cls=0.0361 lr=4.0e-04 | ns22: 0.9694 | macro: 0.9759 | Av=0.9742 Am=0.9750 In=0.9820 Ma=1.0000 [115s] *ns22 *macro
#     Ep11: loss=0.0357 cls=0.0357 lr=3.6e-04 | ns22: 0.9714 | macro: 0.9793 | Av=0.9834 Am=0.9750 In=0.9843 Ma=1.0000 [115s] *ns22 *macro
#     Ep12: loss=0.0352 cls=0.0352 lr=3.3e-04 | ns22: 0.9621 | macro: 0.9766 | Av=0.9715 Am=0.9750 In=0.9745 Ma=1.0000 [115s]
#     Ep13: loss=0.0351 cls=0.0351 lr=2.9e-04 | ns22: 0.9700 | macro: 0.9798 | Av=0.9813 Am=0.9750 In=0.9816 Ma=1.0000 [115s] *macro
#     Ep14: loss=0.0339 cls=0.0339 lr=2.5e-04 | ns22: 0.9698 | macro: 0.9811 | Av=0.9762 Am=0.9750 In=0.9825 Ma=1.0000 [116s] *macro
#     Ep15: loss=0.0334 cls=0.0334 lr=2.1e-04 | ns22: 0.9731 | macro: 0.9799 | Av=0.9857 Am=0.9750 In=0.9807 Ma=1.0000 [114s] *ns22
#     Ep16: loss=0.0336 cls=0.0336 lr=1.7e-04 | ns22: 0.9709 | macro: 0.9833 | Av=0.9742 Am=0.9750 In=0.9833 Ma=1.0000 [115s] *macro
#     Ep17: loss=0.0335 cls=0.0335 lr=1.4e-04 | ns22: 0.9706 | macro: 0.9827 | Av=0.9787 Am=0.9750 In=0.9808 Ma=1.0000 [114s]
#     Ep18: loss=0.0330 cls=0.0330 lr=1.0e-04 | ns22: 0.9748 | macro: 0.9820 | Av=0.9858 Am=0.9750 In=0.9825 Ma=1.0000 [114s] *ns22
#     Ep19: loss=0.0323 cls=0.0323 lr=7.4e-05 | ns22: 0.9736 | macro: 0.9827 | Av=0.9836 Am=0.9750 In=0.9822 Ma=1.0000 [114s]
#     Ep20: loss=0.0323 cls=0.0323 lr=4.9e-05 | ns22: 0.9740 | macro: 0.9822 | Av=0.9828 Am=0.9750 In=0.9826 Ma=1.0000 [114s]
#     Ep21: loss=0.0322 cls=0.0322 lr=2.8e-05 | ns22: 0.9728 | macro: 0.9823 | Av=0.9784 Am=0.9750 In=0.9830 Ma=1.0000 [115s]
#     Early stopping triggered after 5 epochs without improvement.
# --- Starting Greedy Soup based on macro_auc_all ---
# V2 backbone: (1, 768, 8, 10)  (C=768)
#   Model 1 rejected. Temp macro AUC: 0.9830
#   Model 2 rejected. Temp macro AUC: 0.9831
#   Model 3 rejected. Temp macro AUC: 0.9831
#   Model 4 rejected. Temp macro AUC: 0.9831
# V2 backbone: (1, 768, 8, 10)  (C=768)
# W0522 13:44:14.995000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 13:44:14.996000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 13:44:14.996000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# W0522 13:44:14.997000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
# [torch.onnx] Run decomposition...
# [torch.onnx] Run decomposition... ✅
# [torch.onnx] Translate the graph into ONNX...
# [torch.onnx] Translate the graph into ONNX... ✅
# W0522 13:44:18.623000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 13:44:18.624000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 13:44:18.624000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# W0522 13:44:18.625000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
# [torch.onnx] Run decomposition...
# [torch.onnx] Run decomposition... ✅
# [torch.onnx] Translate the graph into ONNX...
# [torch.onnx] Translate the graph into ONNX... ✅
#   ONNX verify: max|diff|=1.942e-03
#   Exported stage1_souped_sed_distill_fold1.onnx (0.4 MB)

# ============================================================
# FOLD 2
# ============================================================

# [Fold 2] Active Dataset Sizes:
#   - focal (FocalDS): 28891 件
#   - sc (ScDS): 593 件
#   - ulsc (ULScDS): 12715 件
#   Streams: {'focal': 28891, 'sc': 593, 'ulsc': 12715}  steps/ep: 659
# V2 backbone: (1, 768, 8, 10)  (C=768)

# [Fold 2] Active Dataset Sizes:
#   - focal (FocalDS): 28891 件
#   - sc (ScDS): 593 件
#   - ulsc (ULScDS): 12715 件
#     Ep00: loss=0.2404 cls=0.2404 lr=1.2e-04 | ns22: 0.5893 | macro: 0.5445 | Av=0.8162 Am=0.4189 In=0.4840 Ma=0.5104 [115s] *ns22 *macro
#     Ep01: loss=0.0595 cls=0.0595 lr=2.1e-04 | ns22: 0.6765 | macro: 0.7936 | Av=0.7824 Am=0.6133 In=0.6342 Ma=0.4792 [114s] *ns22 *macro
#     Ep02: loss=0.0520 cls=0.0520 lr=3.1e-04 | ns22: 0.8238 | macro: 0.8844 | Av=0.8501 Am=0.8713 In=0.7567 Ma=1.0000 [115s] *ns22 *macro
#     Ep03: loss=0.0487 cls=0.0487 lr=4.0e-04 | ns22: 0.8330 | macro: 0.9026 | Av=0.8940 Am=0.9399 In=0.6919 Ma=1.0000 [115s] *ns22 *macro
#     Ep04: loss=0.0460 cls=0.0460 lr=5.0e-04 | ns22: 0.8377 | macro: 0.8601 | Av=0.8935 Am=0.9668 In=0.6923 Ma=1.0000 [114s] *ns22
#     Ep05: loss=0.0430 cls=0.0430 lr=5.0e-04 | ns22: 0.8939 | macro: 0.9056 | Av=0.9076 Am=0.9785 In=0.8238 Ma=1.0000 [115s] *ns22 *macro
#     Ep06: loss=0.0411 cls=0.0411 lr=4.9e-04 | ns22: 0.9248 | macro: 0.9321 | Av=0.9179 Am=0.9858 In=0.8894 Ma=1.0000 [115s] *ns22 *macro
#     Ep07: loss=0.0391 cls=0.0391 lr=4.7e-04 | ns22: 0.9480 | macro: 0.9482 | Av=0.9655 Am=0.9680 In=0.9226 Ma=1.0000 [115s] *ns22 *macro
#     Ep08: loss=0.0381 cls=0.0381 lr=4.5e-04 | ns22: 0.9554 | macro: 0.9547 | Av=0.9709 Am=0.9728 In=0.9313 Ma=1.0000 [115s] *ns22 *macro
#     Ep09: loss=0.0372 cls=0.0372 lr=4.3e-04 | ns22: 0.9595 | macro: 0.9480 | Av=0.9666 Am=0.9816 In=0.9396 Ma=1.0000 [115s] *ns22
#     Ep10: loss=0.0359 cls=0.0359 lr=4.0e-04 | ns22: 0.9619 | macro: 0.9602 | Av=0.9780 Am=0.9686 In=0.9411 Ma=1.0000 [115s] *ns22 *macro
#     Ep11: loss=0.0357 cls=0.0357 lr=3.6e-04 | ns22: 0.9662 | macro: 0.9636 | Av=0.9765 Am=0.9723 In=0.9529 Ma=1.0000 [115s] *ns22 *macro
#     Ep12: loss=0.0349 cls=0.0349 lr=3.3e-04 | ns22: 0.9581 | macro: 0.9692 | Av=0.9713 Am=0.9734 In=0.9373 Ma=1.0000 [114s] *macro
#     Ep13: loss=0.0348 cls=0.0348 lr=2.9e-04 | ns22: 0.9599 | macro: 0.9703 | Av=0.9608 Am=0.9687 In=0.9540 Ma=1.0000 [115s] *macro
#     Ep14: loss=0.0344 cls=0.0344 lr=2.5e-04 | ns22: 0.9673 | macro: 0.9732 | Av=0.9837 Am=0.9712 In=0.9504 Ma=1.0000 [115s] *ns22 *macro
#     Ep15: loss=0.0340 cls=0.0340 lr=2.1e-04 | ns22: 0.9622 | macro: 0.9737 | Av=0.9759 Am=0.9728 In=0.9430 Ma=1.0000 [116s] *macro
#     Ep16: loss=0.0335 cls=0.0335 lr=1.7e-04 | ns22: 0.9623 | macro: 0.9717 | Av=0.9703 Am=0.9697 In=0.9483 Ma=1.0000 [115s]
#     Ep17: loss=0.0331 cls=0.0331 lr=1.4e-04 | ns22: 0.9639 | macro: 0.9722 | Av=0.9781 Am=0.9713 In=0.9451 Ma=1.0000 [115s]
#     Ep18: loss=0.0329 cls=0.0329 lr=1.0e-04 | ns22: 0.9676 | macro: 0.9737 | Av=0.9846 Am=0.9697 In=0.9467 Ma=1.0000 [115s] *ns22
#     Ep19: loss=0.0325 cls=0.0325 lr=7.4e-05 | ns22: 0.9668 | macro: 0.9730 | Av=0.9797 Am=0.9734 In=0.9473 Ma=1.0000 [115s]
#     Ep20: loss=0.0322 cls=0.0322 lr=4.9e-05 | ns22: 0.9681 | macro: 0.9748 | Av=0.9816 Am=0.9755 In=0.9474 Ma=1.0000 [115s] *ns22 *macro
#     Ep21: loss=0.0327 cls=0.0327 lr=2.8e-05 | ns22: 0.9676 | macro: 0.9750 | Av=0.9812 Am=0.9734 In=0.9476 Ma=1.0000 [115s] *macro
#     Ep22: loss=0.0322 cls=0.0322 lr=1.3e-05 | ns22: 0.9690 | macro: 0.9750 | Av=0.9835 Am=0.9734 In=0.9491 Ma=1.0000 [115s] *ns22
#     Ep23: loss=0.0326 cls=0.0326 lr=4.1e-06 | ns22: 0.9691 | macro: 0.9747 | Av=0.9857 Am=0.9739 In=0.9469 Ma=1.0000 [115s] *ns22
#     Ep24: loss=0.0320 cls=0.0320 lr=1.0e-06 | ns22: 0.9686 | macro: 0.9748 | Av=0.9839 Am=0.9739 In=0.9472 Ma=1.0000 [115s]
# --- Starting Greedy Soup based on macro_auc_all ---
# V2 backbone: (1, 768, 8, 10)  (C=768)
#   Model 1 rejected. Temp macro AUC: 0.9749
#   Model 2 rejected. Temp macro AUC: 0.9747
#   Model 3 (Epoch score: 0.9748) added! New macro AUC: 0.9752
#   Model 4 rejected. Temp macro AUC: 0.9750
# V2 backbone: (1, 768, 8, 10)  (C=768)
# W0522 14:32:17.001000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 14:32:17.001000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 14:32:17.002000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# W0522 14:32:17.002000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
# [torch.onnx] Run decomposition...
# [torch.onnx] Run decomposition... ✅
# [torch.onnx] Translate the graph into ONNX...
# [torch.onnx] Translate the graph into ONNX... ✅
# W0522 14:32:19.531000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 14:32:19.532000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 14:32:19.532000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# W0522 14:32:19.533000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
# [torch.onnx] Run decomposition...
# [torch.onnx] Run decomposition... ✅
# [torch.onnx] Translate the graph into ONNX...
# [torch.onnx] Translate the graph into ONNX... ✅
#   ONNX verify: max|diff|=2.519e-03
#   Exported stage1_souped_sed_distill_fold2.onnx (0.4 MB)

# ============================================================
# FOLD 3
# ============================================================

# [Fold 3] Active Dataset Sizes:
#   - focal (FocalDS): 28919 件
#   - sc (ScDS): 590 件
#   - ulsc (ULScDS): 12715 件
#   Streams: {'focal': 28919, 'sc': 590, 'ulsc': 12715}  steps/ep: 659
# V2 backbone: (1, 768, 8, 10)  (C=768)

# [Fold 3] Active Dataset Sizes:
#   - focal (FocalDS): 28919 件
#   - sc (ScDS): 590 件
#   - ulsc (ULScDS): 12715 件
#     Ep00: loss=0.2438 cls=0.2438 lr=1.2e-04 | ns22: 0.6322 | macro: 0.7067 | Av=0.8129 Am=0.4226 In=0.6610 Ma=0.1542 [116s] *ns22 *macro
#     Ep01: loss=0.0595 cls=0.0595 lr=2.1e-04 | ns22: 0.6881 | macro: 0.8214 | Av=0.7976 Am=0.5263 In=0.7253 Ma=0.2500 [116s] *ns22 *macro
#     Ep02: loss=0.0522 cls=0.0522 lr=3.1e-04 | ns22: 0.7959 | macro: 0.8892 | Av=0.7530 Am=0.8031 In=0.8454 Ma=0.5167 [115s] *ns22 *macro
#     Ep03: loss=0.0492 cls=0.0492 lr=4.0e-04 | ns22: 0.8336 | macro: 0.9042 | Av=0.7753 Am=0.8344 In=0.9038 Ma=0.5292 [116s] *ns22 *macro
#     Ep04: loss=0.0468 cls=0.0468 lr=5.0e-04 | ns22: 0.8469 | macro: 0.8970 | Av=0.8234 Am=0.8482 In=0.8868 Ma=0.6292 [115s] *ns22
#     Ep05: loss=0.0438 cls=0.0438 lr=5.0e-04 | ns22: 0.8644 | macro: 0.9181 | Av=0.8544 Am=0.8528 In=0.9093 Ma=0.5208 [115s] *ns22 *macro
#     Ep06: loss=0.0415 cls=0.0415 lr=4.9e-04 | ns22: 0.8696 | macro: 0.9287 | Av=0.8767 Am=0.8264 In=0.9009 Ma=0.6917 [115s] *ns22 *macro
#     Ep07: loss=0.0392 cls=0.0392 lr=4.7e-04 | ns22: 0.8995 | macro: 0.9428 | Av=0.8912 Am=0.8722 In=0.9290 Ma=0.8542 [115s] *ns22 *macro
#     Ep08: loss=0.0382 cls=0.0382 lr=4.5e-04 | ns22: 0.9139 | macro: 0.9498 | Av=0.9217 Am=0.8763 In=0.9389 Ma=0.8417 [115s] *ns22 *macro
#     Ep09: loss=0.0375 cls=0.0375 lr=4.3e-04 | ns22: 0.9339 | macro: 0.9480 | Av=0.9358 Am=0.9306 In=0.9473 Ma=0.8042 [115s] *ns22
#     Ep10: loss=0.0364 cls=0.0364 lr=4.0e-04 | ns22: 0.9216 | macro: 0.9541 | Av=0.9012 Am=0.9289 In=0.9413 Ma=0.8167 [115s] *macro
#     Ep11: loss=0.0355 cls=0.0355 lr=3.6e-04 | ns22: 0.9378 | macro: 0.9533 | Av=0.9150 Am=0.9600 In=0.9532 Ma=0.8917 [115s] *ns22
#     Ep12: loss=0.0353 cls=0.0353 lr=3.3e-04 | ns22: 0.9547 | macro: 0.9660 | Av=0.9444 Am=0.9799 In=0.9514 Ma=0.9000 [115s] *ns22 *macro
#     Ep13: loss=0.0345 cls=0.0345 lr=2.9e-04 | ns22: 0.9482 | macro: 0.9672 | Av=0.9276 Am=0.9739 In=0.9548 Ma=0.9250 [116s] *macro
#     Ep14: loss=0.0341 cls=0.0341 lr=2.5e-04 | ns22: 0.9463 | macro: 0.9646 | Av=0.9148 Am=0.9852 In=0.9466 Ma=0.9792 [116s]
#     Ep15: loss=0.0340 cls=0.0340 lr=2.1e-04 | ns22: 0.9514 | macro: 0.9652 | Av=0.9193 Am=0.9804 In=0.9606 Ma=0.9458 [115s]
#     Ep16: loss=0.0340 cls=0.0340 lr=1.7e-04 | ns22: 0.9646 | macro: 0.9691 | Av=0.9471 Am=0.9824 In=0.9681 Ma=1.0000 [115s] *ns22 *macro
#     Ep17: loss=0.0338 cls=0.0338 lr=1.4e-04 | ns22: 0.9651 | macro: 0.9712 | Av=0.9473 Am=0.9851 In=0.9653 Ma=0.9875 [115s] *ns22 *macro
#     Ep18: loss=0.0333 cls=0.0333 lr=1.0e-04 | ns22: 0.9615 | macro: 0.9702 | Av=0.9441 Am=0.9858 In=0.9623 Ma=0.9583 [115s]
#     Ep19: loss=0.0328 cls=0.0328 lr=7.4e-05 | ns22: 0.9659 | macro: 0.9701 | Av=0.9516 Am=0.9876 In=0.9676 Ma=0.9667 [115s] *ns22
#     Ep20: loss=0.0327 cls=0.0327 lr=4.9e-05 | ns22: 0.9648 | macro: 0.9714 | Av=0.9482 Am=0.9910 In=0.9610 Ma=0.9750 [115s] *macro
#     Ep21: loss=0.0325 cls=0.0325 lr=2.8e-05 | ns22: 0.9671 | macro: 0.9723 | Av=0.9522 Am=0.9892 In=0.9634 Ma=0.9917 [115s] *ns22 *macro
#     Ep22: loss=0.0322 cls=0.0322 lr=1.3e-05 | ns22: 0.9643 | macro: 0.9712 | Av=0.9434 Am=0.9909 In=0.9624 Ma=0.9917 [115s]
#     Ep23: loss=0.0326 cls=0.0326 lr=4.1e-06 | ns22: 0.9670 | macro: 0.9726 | Av=0.9500 Am=0.9920 In=0.9632 Ma=0.9917 [115s] *macro
#     Ep24: loss=0.0321 cls=0.0321 lr=1.0e-06 | ns22: 0.9666 | macro: 0.9723 | Av=0.9509 Am=0.9915 In=0.9617 Ma=0.9917 [115s]
# --- Starting Greedy Soup based on macro_auc_all ---
# V2 backbone: (1, 768, 8, 10)  (C=768)
#   Model 1 (Epoch score: 0.9723) added! New macro AUC: 0.9726
#   Model 2 (Epoch score: 0.9723) added! New macro AUC: 0.9726
#   Model 3 rejected. Temp macro AUC: 0.9725
#   Model 4 rejected. Temp macro AUC: 0.9723
# V2 backbone: (1, 768, 8, 10)  (C=768)
# W0522 15:20:29.605000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 15:20:29.606000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 15:20:29.606000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# W0522 15:20:29.607000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
# [torch.onnx] Run decomposition...
# [torch.onnx] Run decomposition... ✅
# [torch.onnx] Translate the graph into ONNX...
# [torch.onnx] Translate the graph into ONNX... ✅
# W0522 15:20:32.096000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 15:20:32.097000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 15:20:32.097000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# W0522 15:20:32.098000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
# [torch.onnx] Run decomposition...
# [torch.onnx] Run decomposition... ✅
# [torch.onnx] Translate the graph into ONNX...
# [torch.onnx] Translate the graph into ONNX... ✅
#   ONNX verify: max|diff|=4.680e-03
#   Exported stage1_souped_sed_distill_fold3.onnx (0.4 MB)

# ============================================================
# FOLD 4
# ============================================================

# [Fold 4] Active Dataset Sizes:
#   - focal (FocalDS): 28916 件
#   - sc (ScDS): 587 件
#   - ulsc (ULScDS): 12715 件
#   Streams: {'focal': 28916, 'sc': 587, 'ulsc': 12715}  steps/ep: 659
# V2 backbone: (1, 768, 8, 10)  (C=768)

# [Fold 4] Active Dataset Sizes:
#   - focal (FocalDS): 28916 件
#   - sc (ScDS): 587 件
#   - ulsc (ULScDS): 12715 件
#     Ep00: loss=0.2527 cls=0.2527 lr=1.2e-04 | ns22: 0.6231 | macro: 0.7805 | Av=0.8321 Am=0.3493 In=0.5084 Ma=0.8241 [115s] *ns22 *macro
#     Ep01: loss=0.0593 cls=0.0593 lr=2.1e-04 | ns22: 0.7154 | macro: 0.8711 | Av=0.8089 Am=0.7007 In=0.6155 Ma=0.7593 [116s] *ns22 *macro
#     Ep02: loss=0.0519 cls=0.0519 lr=3.1e-04 | ns22: 0.7576 | macro: 0.8923 | Av=0.6245 Am=0.8108 In=0.8392 Ma=0.7963 [115s] *ns22 *macro
#     Ep03: loss=0.0488 cls=0.0488 lr=4.0e-04 | ns22: 0.9023 | macro: 0.9301 | Av=0.8748 Am=0.8606 In=0.9233 Ma=1.0000 [115s] *ns22 *macro
#     Ep04: loss=0.0463 cls=0.0463 lr=5.0e-04 | ns22: 0.9120 | macro: 0.9142 | Av=0.8693 Am=0.8199 In=0.9671 Ma=0.9630 [115s] *ns22
#     Ep05: loss=0.0436 cls=0.0436 lr=5.0e-04 | ns22: 0.8908 | macro: 0.9280 | Av=0.8525 Am=0.8459 In=0.9273 Ma=0.9815 [115s]
#     Ep06: loss=0.0414 cls=0.0414 lr=4.9e-04 | ns22: 0.9077 | macro: 0.8988 | Av=0.8395 Am=0.8606 In=0.9742 Ma=0.9444 [116s]
#     Ep07: loss=0.0394 cls=0.0394 lr=4.7e-04 | ns22: 0.9254 | macro: 0.9337 | Av=0.8904 Am=0.8548 In=0.9743 Ma=0.9537 [115s] *ns22 *macro
#     Ep08: loss=0.0384 cls=0.0384 lr=4.5e-04 | ns22: 0.9006 | macro: 0.9356 | Av=0.8165 Am=0.8240 In=0.9835 Ma=0.9537 [115s] *macro
#     Ep09: loss=0.0371 cls=0.0371 lr=4.3e-04 | ns22: 0.9491 | macro: 0.9413 | Av=0.9330 Am=0.8420 In=0.9899 Ma=0.9722 [115s] *ns22 *macro
#     Ep10: loss=0.0366 cls=0.0366 lr=4.0e-04 | ns22: 0.9400 | macro: 0.9470 | Av=0.9103 Am=0.8952 In=0.9732 Ma=0.9537 [115s] *macro
#     Ep11: loss=0.0359 cls=0.0359 lr=3.6e-04 | ns22: 0.9435 | macro: 0.9479 | Av=0.9087 Am=0.8954 In=0.9797 Ma=0.9907 [115s] *macro
#     Ep12: loss=0.0351 cls=0.0351 lr=3.3e-04 | ns22: 0.9368 | macro: 0.9417 | Av=0.8952 Am=0.8891 In=0.9812 Ma=0.9630 [115s]
#     Ep13: loss=0.0346 cls=0.0346 lr=2.9e-04 | ns22: 0.9649 | macro: 0.9428 | Av=0.9560 Am=0.9025 In=0.9863 Ma=1.0000 [115s] *ns22
#     Ep14: loss=0.0343 cls=0.0343 lr=2.5e-04 | ns22: 0.9617 | macro: 0.9488 | Av=0.9595 Am=0.9189 In=0.9736 Ma=0.9630 [115s] *macro
#     Ep15: loss=0.0337 cls=0.0337 lr=2.1e-04 | ns22: 0.9654 | macro: 0.9538 | Av=0.9417 Am=0.9558 In=0.9851 Ma=0.9907 [115s] *ns22 *macro
#     Ep16: loss=0.0333 cls=0.0333 lr=1.7e-04 | ns22: 0.9748 | macro: 0.9494 | Av=0.9707 Am=0.9660 In=0.9778 Ma=0.9907 [115s] *ns22
#     Ep17: loss=0.0334 cls=0.0334 lr=1.4e-04 | ns22: 0.9672 | macro: 0.9465 | Av=0.9527 Am=0.9577 In=0.9780 Ma=0.9907 [115s]
#     Ep18: loss=0.0329 cls=0.0329 lr=1.0e-04 | ns22: 0.9722 | macro: 0.9486 | Av=0.9651 Am=0.9516 In=0.9810 Ma=0.9907 [115s]
#     Ep19: loss=0.0326 cls=0.0326 lr=7.4e-05 | ns22: 0.9742 | macro: 0.9487 | Av=0.9610 Am=0.9843 In=0.9790 Ma=0.9907 [115s]
#     Ep20: loss=0.0322 cls=0.0322 lr=4.9e-05 | ns22: 0.9735 | macro: 0.9492 | Av=0.9606 Am=0.9759 In=0.9802 Ma=0.9907 [115s]
#     Early stopping triggered after 5 epochs without improvement.
# --- Starting Greedy Soup based on macro_auc_all ---
# V2 backbone: (1, 768, 8, 10)  (C=768)
#   Model 1 rejected. Temp macro AUC: 0.9531
#   Model 2 rejected. Temp macro AUC: 0.9505
#   Model 3 rejected. Temp macro AUC: 0.9517
#   Model 4 rejected. Temp macro AUC: 0.9527
# V2 backbone: (1, 768, 8, 10)  (C=768)
# W0522 16:00:54.802000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 16:00:54.803000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 16:00:54.803000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# W0522 16:00:54.804000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
# [torch.onnx] Run decomposition...
# [torch.onnx] Run decomposition... ✅
# [torch.onnx] Translate the graph into ONNX...
# [torch.onnx] Translate the graph into ONNX... ✅
# W0522 16:00:57.301000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 16:00:57.302000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
# W0522 16:00:57.303000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# W0522 16:00:57.303000 2696 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
# [torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
# [torch.onnx] Run decomposition...
# [torch.onnx] Run decomposition... ✅
# [torch.onnx] Translate the graph into ONNX...
# [torch.onnx] Translate the graph into ONNX... ✅
#   ONNX verify: max|diff|=1.271e-03
#   Exported stage1_souped_sed_distill_fold4.onnx (0.4 MB)


* ns22: サイト「S22」を除外した検証用Soundscapeデータにおける、予測ブレンド（Clip予測とFrame-max予測の平均）のMacro AUCスコアです。S22はラベルノイズが多い既知のサイトであるため、これを除外したこのスコアが主力の評価指標として機能しています。  

* 分類（Taxonomy）ごとのAUCスコア: S22を除外したデータにおける、生物分類ごとのMacro AUCスコアです。  

    * Av: 鳥類（Aves）

    * Am: 両生類（Amphibia）

    * In: 昆虫類（Insecta）

    * Ma: 哺乳類（Mammalia）

# S7 — OOF Evaluation

In [ ]:
# =================================================================
# S7 -- EVALUATION SUMMARY
# =================================================================

if MODE != "train":
    print("Skipping evaluation (MODE='infer')")
else:

    has = ~np.isnan(oof_ns22[:, 0])
    if has.sum() > 0:
        r_all = full_eval(Y_SC[has], oof_ns22[has], non_s22_mask_sc[has], TAXON_MASKS)
        print("=" * 60)
        print("OOF RESULTS (best-ns22 checkpoints)")
        print("=" * 60)
        print(f"  macro AUC (all):        {r_all['macro_auc_all']:.4f}")
        print(f"  macro AUC (non-S22):    {r_all['non_s22_macro']:.4f}")
        for t in ["Aves", "Amphibia", "Insecta", "Mammalia"]:
            print(f"    {t:<12}: {r_all.get(f'non_s22_{t}', float('nan')):.4f}")

    # Per-epoch progression
    print("\nPer-epoch pooled non-S22 AUC:")
    fold_true, fold_ns22_m = {}, {}
    for fk in range(N_FOLDS):
        vm = sc_cache_meta["fold"].values == fk
        fold_true[fk] = Y_SC[vm]
        fold_ns22_m[fk] = non_s22_mask_sc[vm]

    n_eps = [len(all_hist[k]["val_preds"]) for k in range(N_FOLDS) if k in all_hist]
    max_ep = min(n_eps) if n_eps else 0
    for ep in range(max_ep):
        pp = np.concatenate([all_hist[k]["val_preds"][ep] for k in range(N_FOLDS) if k in all_hist])
        pt = np.concatenate([fold_true[k] for k in range(N_FOLDS) if k in all_hist])
        pm = np.concatenate([fold_ns22_m[k] for k in range(N_FOLDS) if k in all_hist])
        ns, _ = compute_macro_auc(pt, pp, mask=pm)
        print(f"  Ep{ep:02d}: {ns:.4f}")

In [ ]:
import time
from google.colab import runtime

# 1時間（X秒）待機してから終了する場合
time.sleep(10)
runtime.unassign()